# LLM-Powered Stock Trading Agent

## Paper Trading with LLM Decision Making, News Sentiment & RAG Memory

---

**Student Information**

Name: Andrew Bugbee

Student ID: 901017334

> **Update (Oct 11):** Alpha Vantage made news (`NEWS_SENTIMENT`) and `MACD` premium-only. Section 1.9 now gets news from **Alpaca**, and Task 2 computes MACD from **yfinance** prices. RSI and GLOBAL_QUOTE still use Alpha Vantage.

---

### Assignment Overview

In this assignment, you will build components of an **AI-powered stock trading agent** that:
1. Uses real-time financial news and sentiment data to inform decisions
2. Extends data sources with custom retrieval functions (MCP Styled Tools)
3. Learns from past trades using a RAG memory system (BM25 + Reflection)
4. Executes paper trades via the Alpaca brokerage API

### Task Breakdown

| Task | Topic | Status |
|------|-------|--------|
| **Task 1** | Environment, Infrastructure & Core Logic | TODO |
| **Task 2** | MCP Tool Functions (Additional Data Sources) | TODO |
| **Task 3** | RAG Memory System (BM25 + Reflection) | TODO |
| **Task 4** | SFT Fine-tuning with LoRA | TODO |
| **Task 5** | Full Trading Loop Integration | TODO |


---
## Task 1: Environment, Infrastructure & Core Logic

### Grading criteria

- [ ] `llm_agent_decide()` (1.12) runs the multi-turn tool loop and returns 4 values. *Evidence: 1.15 output (rerun after Task 2) shows, turn by turn, the tool called → the first lines of its result → the model's final raw text → a decision that can be executed.*
- [ ] Most days in the 4a backtest have a real decision: the model calls a tool and gives at least one valid buy or sell, rather than the code falling back to a default. *Evidence: 4a output: `Episodes` (days logged) out of the backtest days. Run 4a before 4c, and keep the turn-by-turn log.*
- If your agent can HOLD, explain its HOLD decisions in your report with 2–3 examples from your log.


### 1.1 Install Dependencies

In [1]:
!pip install -q transformers accelerate alpaca-py sentence-transformers faiss-cpu requests peft trl datasets yfinance python-dotenv exchange-calendars
# Colab's preinstalled torchao is too old for current peft (Task 4 LoRA fails). It is not used here.
!pip uninstall -y torchao



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\albug\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


### 1.2 Imports

In [2]:
import os
import json
import time
import datetime
import requests
from typing import Dict, Any, List, Optional
from dotenv import load_dotenv

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce


print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Data
import pandas as pd
import yfinance as yf

# Fine-tuning (Task 5)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

c:\Users\albug\anaconda3\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful!
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Ti


### 1.3 Set Up Local Cache

News data will be cached to the kernel's working directory so you don't re-fetch it every run.


In [3]:
from pathlib import Path

cache_path = Path.cwd() / ".cache" / "stock_agent"
cache_path.mkdir(parents=True, exist_ok=True)

CACHE_DIR = str(cache_path)
print(f"Cache directory: {CACHE_DIR}")

Cache directory: c:\Users\albug\VS-Projects\LLM-trading-agent\.cache\stock_agent


### 1.4 [TODO] Configuration & API Keys

---

#### Step 1 — Add your API keys

| Secret name | Where to get it |
|-------------|-----------------|
| `ALPACA_API_KEY` | [alpaca.markets](https://alpaca.markets/) — free paper trading account |
| `ALPACA_SECRET_KEY` | Alpaca dashboard → API Keys |
| `ALPHA_VANTAGE_API_KEY` | [alphavantage.co](https://www.alphavantage.co/support/#api-key) — free tier (25 calls/day) |

---

#### Step 2 — Understand the configurable parameters

The bottom of the code cell below is yours to edit. Here is what each variable does:

**Trading Parameters**

| Variable | Default | Effect |
|----------|---------|--------|
| `MAX_TRADE_VALUE` | `$10,000` | Maximum estimated dollar value of each buy or sell. Whole-share quantities are reduced to fit; market fills can differ from the reference price. |
| `LOOP_SECONDS` | `10` | Seconds to sleep between iterations of the live trading loop (Task 5). Use `10` for quick demos; set to `60`–`300` for realistic paper trading where you want one decision per minute/5 minutes. |
| `ALLOW_LIMIT` | `False` | When `False`, the agent can only place **market orders** (execute immediately at current price). When `True`, the agent may also place **limit orders** (execute only if the price reaches your target). Start with `False` — it keeps the order logic simpler. |

**Memory & Reflection Parameters** *(Task 3 and onward)*

| Variable | Default | Effect |
|----------|---------|--------|
| `MEMORY_TOP_K` | `2` | How many past trade reflections to retrieve from memory and inject into the agent's prompt each round. `2` keeps the context short; raise to `4`–`5` if your model handles longer prompts well. Too high and it crowds out the current market state. |
| `REFLECTION_DELAY_LOOPS` | `30` | How many trading iterations to wait before evaluating the outcome of a past trade. This gives the market time to move before you ask "was that a good trade?" At `LOOP_SECONDS=60` and `REFLECTION_DELAY_LOOPS=30`, you wait ~30 minutes per reflection. Lower it to `5`–`10` for backtesting; keep it at 30+ for live trading. |

---

#### Step 3 — Optionally extend the tradeable universe

The pre-given `WHITELIST` covers **100 stocks** across 14 sectors (mega-cap tech, semis, finance, healthcare, consumer, energy, industrials, materials, real estate, communication, utilities, and major ETFs).

**The agent does NOT fetch news for all 100 at once.** Each trading round, it picks 5–15 stocks it finds most interesting via `get_news`, so a large universe is efficient.

You have three options:

**Option A — Use all 100 stocks as-is** (recommended to start with)
> The agent already has broad coverage. Just run the notebook as provided.

**Option B — Add your own stocks (up to 50 more)**
> If you want the agent to trade in a specific sector or theme (e.g. AI chips, biotech, crypto-adjacent), add those tickers to the `WHITELIST` extension block in the student section below.
>
> Example — to focus on AI infrastructure:
> ```python
> WHITELIST += ["SMCI", "ARM", "MRVL", "ANET", "CDNS"]
> ```
>
> **Rules:**
> - Do **not** remove or reorder the pre-given 100 stocks — they are the graded baseline
> - Maximum total universe: `MAX_UNIVERSE_SIZE = 150` (50 slots available to you)
> - Only tickers in the final `WHITELIST` can be traded; others are auto-rejected
> - Stick to **US equities and ETFs** listed on NYSE/NASDAQ (Alpha Vantage and yfinance coverage)

**Option C — Let the agent discover stocks from news (advanced)**
> Instead of pre-defining target stocks, you implement a **market discovery tool** in Task 2
> that fetches broad market news *without* specifying any ticker symbols. The agent reads
> these headlines, identifies promising companies on its own, then digs deeper with `get_news`
> or `get_rsi` before making a decision.
>
> Use the Alpaca News API from Section 1.9 **without** `symbols`: it returns the latest
> market-wide news, and each article has a `symbols` list with the companies mentioned, so the
> agent gets both the story and the relevant tickers in one call. Alpaca has no topic filter and
> no sentiment scores; to keep a `topic` argument, filter the headlines by keyword yourself.
> (Alpha Vantage's `NEWS_SENTIMENT` is now premium-only.)
>
> **How to wire it up in Task 2:**
> ```python
> def search_market_news(topic: str = "technology") -> str:
>     """Fetch broad market news; returns top mentioned tickers + headlines."""
>     # Call the Alpaca News API without symbols= (latest market-wide news)
>     # Optional: keep only articles whose headline/summary mentions topic
>     # Count each article's symbols; return formatted string the agent can read
>     raise NotImplementedError("TODO")
>
> register_tool(
>     "search_market_news",
>     "Fetch the latest market-wide news (no ticker needed) and the most "
>     "mentioned tickers. Optional topic keyword, e.g. technology, earnings, ipo.",
>     {"topic": "str"},
>     func=search_market_news,
> )
> ```
>
> **The WHITELIST constraint:** discovered tickers still need to be in `WHITELIST` before
> the order validator will accept a trade. Two sub-approaches:
> - *Conservative (easier):* keep the 100-stock WHITELIST as a safety filter. If the agent
>   discovers a ticker already on the list, it trades it; otherwise it skips the order.
>   No extra code needed.
> - *Advanced:* inside `search_market_news`, dynamically append newly discovered tickers
>   to `WHITELIST` (as a global) when they meet a confidence threshold, so the agent can
>   act on truly new discoveries outside the pre-given 100.
>
> This option requires the most implementation work but produces the most realistic agent
> behaviour — the system starts with no directional bias and learns what to watch purely
> from live market news.



In [4]:
# Retrieving evironment secrets from .env
load_dotenv()
ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")
ALPHA_VANTAGE_API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
def parse_paper_setting(value):
    """Parse an explicit boolean; an unset value defaults to paper mode."""
    if value is None:
        return True
    normalized = value.strip().lower()
    if normalized in ("true", "1", "yes", "on"):
        return True
    if normalized in ("false", "0", "no", "off"):
        return False
    raise ValueError("ALPACA_PAPER must be true/false, 1/0, yes/no, or on/off")


ALPACA_PAPER = parse_paper_setting(os.getenv("ALPACA_PAPER"))

if not ALPACA_API_KEY:
    raise ValueError("Alpaca credentials missing, check .env")

# =============================================================================
# DO NOT MODIFY — Pre-given Tradeable Universe (100 stocks, 14 sectors)
# ─ The agent's tools (get_news, get_rsi, etc.) query any subset of these.
# ─ Orders for symbols NOT in WHITELIST are automatically rejected.
# ─ Do not remove or reorder these 100 entries.
# =============================================================================
WHITELIST = [
    # Mega-cap Tech (10)
    "AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "NFLX", "ADBE", "ORCL",
    # Semiconductors (8)
    "AMD", "INTC", "AVGO", "QCOM", "MU", "AMAT", "LRCX", "KLAC",
    # Other Tech / Software (8)
    "CRM", "UBER", "PLTR", "SNOW", "CSCO", "IBM", "RBLX", "SNAP",
    # Finance (10)
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "PYPL", "AXP", "BLK",
    # Healthcare (10)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "LLY", "AMGN", "GILD", "BMY", "CVS",
    # Consumer Staples (6)
    "WMT", "KO", "PEP", "PG", "COST", "MO",
    # Consumer Discretionary (8)
    "MCD", "NKE", "DIS", "SBUX", "TGT", "HD", "F", "GM",
    # Energy (6)
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC",
    # Industrials (8)
    "CAT", "BA", "GE", "HON", "UPS", "FDX", "DE", "RTX",
    # Materials & Commodities (6)
    "LIN", "NEM", "FCX", "DOW", "APD", "NUE",
    # Real Estate (5)
    "AMT", "PLD", "EQIX", "O", "SPG",
    # Communication (5)
    "T", "VZ", "CMCSA", "TMUS", "CHTR",
    # Utilities (1)
    "NEE",
    # Broad ETFs (4)
    "SPY", "QQQ", "IWM", "DIA",
    # Sector ETFs (5)
    "XLF", "XLK", "XLE", "XLV", "XLI",
]

# =============================================================================
# STUDENT CONFIGURATION — Edit the values in this section
# =============================================================================

# ── Universe Extension ────────────────────────────────────────────────────────
# Want to focus on a specific sector or theme? Add your tickers here.
# The agent discovers opportunities via get_news and your MCP tools (Task 2).
# Rules: do NOT remove the 100 above | max total = MAX_UNIVERSE_SIZE = 150
# Example: WHITELIST += ["SMCI", "ARM", "MRVL", "ANET"]
WHITELIST += [
    # Add your symbols here:

]

# ── Trading Parameters ────────────────────────────────────────────────────────
MAX_TRADE_VALUE = 10000.0  # Maximum estimated dollar value per buy or sell.
MAX_REFERENCE_AGE_SECONDS = 300  # Reject stale live prices when sizing orders.

LOOP_SECONDS = 10           # Seconds between live trading iterations (Task 5).
                            # Demo: 10s  |  Realistic paper trading: 60–300s.

ALLOW_LIMIT = False         # False = market orders only (always fills immediately).
                            # True  = agent may also place limit orders at a target price.

# ── Memory & Reflection Parameters ───────────────────────────────────────────
MEMORY_MIN_SIMILARITY = 0.35  # Minimum cosine similarity for recalled memories.
MEMORY_TOP_K = 2            # Past reflections injected into agent context each round.
                            # Higher = richer context. Too high crowds out market data.
                            # Suggested range: 1–5. Default: 2.

REFLECTION_DELAY_LOOPS = 10 # Loops to wait before scoring a past trade's outcome.
                            # At LOOP_SECONDS=60 → delay=30 means ~30 min lookback.
                            # Backtest: use 5–10.  Live trading: use 20–50.

# =============================================================================
# DO NOT MODIFY — Infrastructure constants
# =============================================================================
MAX_UNIVERSE_SIZE      = 150   # Hard cap on WHITELIST length
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
NEWS_LIMIT             = 5
MODEL_NAME             = "Qwen/Qwen2.5-1.5B-Instruct"

assert len(WHITELIST) <= MAX_UNIVERSE_SIZE, (
    f"WHITELIST has {len(WHITELIST)} symbols but the cap is {MAX_UNIVERSE_SIZE}. "
    "Remove some symbols from your extension block."
)
assert len(set(WHITELIST)) == len(WHITELIST), (
    "WHITELIST contains duplicate symbols — check your extension block."
)

print("\nConfiguration loaded:")
print(f"  Model:          {MODEL_NAME}")
print(f"  Universe:       {len(WHITELIST)} stocks  (cap: {MAX_UNIVERSE_SIZE})")
print(f"  Max trade value: ${MAX_TRADE_VALUE:,.2f} (estimated)")
print(f"  Loop interval:  {LOOP_SECONDS}s")
print(f"  Limit orders:   {ALLOW_LIMIT}")
print(f"  Memory top-k:   {MEMORY_TOP_K}")
print(f"  Reflect delay:  {REFLECTION_DELAY_LOOPS} loops")
print(f"  Paper requested: {ALPACA_PAPER} (endpoint not yet verified)")



Configuration loaded:
  Model:          Qwen/Qwen2.5-1.5B-Instruct
  Universe:       100 stocks  (cap: 150)
  Max trade value: $10,000.00 (estimated)
  Loop interval:  10s
  Limit orders:   False
  Memory top-k:   2
  Reflect delay:  10 loops
  Paper requested: True (endpoint not yet verified)


### 1.5 Utility Functions

LLMs return **raw text**, not structured data. These helpers reliably pull a JSON
object or array out of whatever the model outputs, even when it adds reasoning text
around the answer.

```
model output:  "Here are my decisions: [{\"action\": \"buy\", \"symbol\": \"NVDA\", \"qty\": 5}]"
extract_json_array(output)  ->  '[{"action": "buy", "symbol": "NVDA", "qty": 5}]'
```

Used internally by the agent loop — you don’t need to call these directly.


In [5]:
# =============================================================================
# DO NOT MODIFY - JSON Extraction Utilities
# =============================================================================

def extract_last_json(text: str):
    """Extract the last valid JSON object {...} from text."""
    i = text.rfind("{")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "{":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


def extract_json_array(text: str):
    """Extract the first valid JSON array [...] from text."""
    i = text.find("[")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "[":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "]":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


# Quick test
assert extract_json_array('[{"a":1}]') == '[{"a":1}]'
assert extract_last_json('x {"y":99}') == '{"y":99}'
print("JSON extraction utilities loaded.")

JSON extraction utilities loaded.


### 1.6 Load LLM Model

Loads **Qwen2.5-1.5B-Instruct** on T4 GPU (~30 seconds).


In [6]:
# =============================================================================
# DO NOT MODIFY - Model Loading
# =============================================================================

def load_model(model_name=MODEL_NAME):
    print(f"Loading model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map={"": 0} if torch.cuda.is_available() else None,
    )
    model.eval()
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded on {device} (dtype={dtype})")
    return tok, model

tok, model = load_model()

Loading model: Qwen/Qwen2.5-1.5B-Instruct ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 121.78it/s]

Model loaded on GPU (dtype=torch.float16)


### 1.7 Quick Model Test

In [7]:
test_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is paper trading in one sentence?"},
]
test_prompt = tok.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tok(test_prompt, return_tensors="pt", add_special_tokens=False)
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, eos_token_id=tok.eos_token_id)
response = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
print("LLM Response:", response[:300])

LLM Response: Paper trading involves simulating trades using simulated or historical data to test trading strategies without risking real money.


### 1.8 Alpaca Trading Client


In [8]:
# =============================================================================
# DO NOT MODIFY - Trading Infrastructure
# =============================================================================

PAPER_TRADING_URL = "https://paper-api.alpaca.markets"


def verify_paper_account(tc: TradingClient) -> str:
    """Require Alpaca's paper endpoint and an authenticated account ID."""
    if ALPACA_PAPER is not True:
        raise RuntimeError("Paper trading must be enabled before placing orders")
    base_url = getattr(tc, "_base_url", None)
    base_url = getattr(base_url, "value", base_url)
    if not isinstance(base_url, str) or base_url.rstrip("/") != PAPER_TRADING_URL:
        raise RuntimeError("Trading client is not configured for Alpaca's paper endpoint")
    account = tc.get_account()
    account_id = getattr(account, "id", None)
    if not account_id:
        raise RuntimeError("Could not verify the authenticated paper account")
    return str(account_id)


def trading_client() -> TradingClient:
    if ALPACA_PAPER is not True:
        raise RuntimeError("This notebook permits paper trading only; set ALPACA_PAPER=true")
    return TradingClient(api_key=ALPACA_API_KEY, secret_key=ALPACA_SECRET_KEY, paper=True)


def get_state(tc: TradingClient) -> Dict[str, Any]:
    acct = tc.get_account()
    positions = tc.get_all_positions()
    pos = []
    for p in positions:
        pos.append({
            "symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "avg_entry_price": float(p.avg_entry_price) if p.avg_entry_price is not None else None,
        })
    return {
        "cash": float(acct.cash),
        "equity": float(acct.equity),
        "buying_power": float(acct.buying_power),
        "positions": pos,
        "allowed_symbols": WHITELIST,
        "max_trade_value": MAX_TRADE_VALUE,
        "timestamp_unix": int(time.time()),
    }


def cap_trade_quantity(quantity, reference_price):
    """Return whole shares within the per-order dollar budget, or zero if invalid."""
    from decimal import Decimal, InvalidOperation, ROUND_FLOOR
    if not isinstance(quantity, int) or isinstance(quantity, bool) or quantity <= 0:
        return 0
    try:
        price = Decimal(str(reference_price))
        limit = Decimal(str(MAX_TRADE_VALUE))
        if not price.is_finite() or price <= 0 or not limit.is_finite() or limit <= 0:
            return 0
        affordable = int((limit / price).to_integral_value(rounding=ROUND_FLOOR))
        return min(quantity, max(0, affordable))
    except (InvalidOperation, ValueError, TypeError, OverflowError):
        return 0


def get_trade_reference_price(symbol):
    """Use a recent Alpaca snapshot price for live order sizing, or return None."""
    from decimal import Decimal, InvalidOperation
    if not isinstance(symbol, str) or symbol not in WHITELIST:
        return None
    try:
        response = requests.get(
            f"https://data.alpaca.markets/v2/stocks/{symbol}/snapshot",
            headers={"APCA-API-KEY-ID": ALPACA_API_KEY, "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY},
            timeout=15,
        )
        if response.status_code != 200:
            return None
        data = response.json()
        now = datetime.datetime.now(datetime.timezone.utc)
        candidates = []
        for key, fields in (("latestTrade", ("p",)), ("latestQuote", ("ap", "bp"))):
            record = data.get(key) or {}
            stamp = record.get("t")
            if not isinstance(stamp, str):
                continue
            try:
                observed = datetime.datetime.fromisoformat(stamp.replace("Z", "+00:00"))
                if observed.tzinfo is None:
                    continue
                age = (now - observed.astimezone(datetime.timezone.utc)).total_seconds()
                if not (-30 <= age <= MAX_REFERENCE_AGE_SECONDS):
                    continue
                for field in fields:
                    try:
                        value = Decimal(str(record.get(field)))
                        if value.is_finite() and value > 0:
                            candidates.append(value)
                    except (InvalidOperation, TypeError, ValueError):
                        continue
            except (InvalidOperation, TypeError, ValueError, OverflowError):
                continue
        return max(candidates) if candidates else None
    except Exception as exc:
        print(f"  [Trade Value] Price unavailable for {symbol}: {type(exc).__name__}")
        return None


def submit_order(tc: TradingClient, d: Dict[str, Any], expected_account_id: str = None) -> Dict[str, Any]:
    account_id = verify_paper_account(tc)
    if expected_account_id is not None and account_id != expected_account_id:
        raise RuntimeError("Paper account changed during the trading loop")
    symbol = d.get("symbol")
    action = d.get("action")
    requested_qty = d.get("qty")
    if symbol not in WHITELIST or action not in ("buy", "sell"):
        raise ValueError("Invalid trading symbol or action")
    if d.get("order_type") not in ("market", "limit"):
        raise ValueError("Unsupported order type")
    reference_price = get_trade_reference_price(symbol)
    if d.get("order_type") == "limit":
        from decimal import Decimal, InvalidOperation
        try:
            limit_price = Decimal(str(d.get("limit_price")))
            if not limit_price.is_finite() or limit_price <= 0 or reference_price is None:
                raise ValueError("Invalid price")
            reference_price = max(Decimal(str(reference_price)), limit_price)
        except (InvalidOperation, ValueError, TypeError):
            raise ValueError("A valid reference price and limit price are required")
    quantity = cap_trade_quantity(requested_qty, reference_price)
    if quantity <= 0:
        raise ValueError("No whole share fits the trade-value limit or a fresh price is unavailable")
    if quantity < requested_qty:
        print(f"  [Trade Value] Reduced {symbol} from {requested_qty} to {quantity} shares")
    d["qty"] = quantity
    side = OrderSide.BUY if action == "buy" else OrderSide.SELL
    if d["order_type"] == "limit":
        req = LimitOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY, limit_price=str(d["limit_price"]),
        )
    else:
        req = MarketOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY,
        )
    order = tc.submit_order(order_data=req)

    def _to_float(v):
        if v is None:
            return None
        try:
            fv = float(v)
            return fv if fv == fv and fv not in (float("inf"), float("-inf")) else None
        except (ValueError, TypeError):
            return None

    filled_qty = _to_float(getattr(order, "filled_qty", None))
    filled_avg_price = _to_float(getattr(order, "filled_avg_price", None))

    return {
        "id": str(order.id),
        "status": str(order.status),
        "symbol": d["symbol"],
        "side": d["action"],
        "qty": d["qty"],
        "filled_qty": filled_qty,
        "filled_avg_price": filled_avg_price,
    }

# --- Test connection ---
tc = trading_client()
verify_paper_account(tc)
state = get_state(tc)
print("Alpaca paper endpoint and account verified!")
print(f"  Equity:       ${state['equity']:,.2f}")
print(f"  Cash:         ${state['cash']:,.2f}")
print(f"  Buying Power: ${state['buying_power']:,.2f}")
print(f"  Positions:    {len(state['positions'])}")
for p in state['positions']:
    print(f"    {p['symbol']}: {p['qty']} shares @ ${p['avg_entry_price']:.2f}")

Alpaca paper endpoint and account verified!
  Equity:       $100,000.00
  Cash:         $100,000.00
  Buying Power: $400,000.00
  Positions:    0


### 1.9 [TODO] News Retrieval (Cached by Request Parameters)

`get_news_for_symbols()` fetches financial news from the **Alpaca News API** (same keys as your
paper trading account) and caches results locally (or to Google Drive) keyed by canonical symbols,
date, limit, request mode, and historical decision cutoff when supplied.

> **Update (Oct 11):** this section used Alpha Vantage `NEWS_SENTIMENT`, which is now premium-only.
> If you already implemented it with Alpha Vantage, only the API call and the parsing change.
> Delete any `news_*.json` in `CACHE_DIR` that says "No news data available".

**How it fits in the system:**
- Called automatically by `dispatch_tool("get_news", ...)` when the trading agent
  requests news during its decision loop
- The backtest does not pre-fetch news: the agent's first `get_news` call for a specific request fetches
  from the API and is cached; later calls for identical parameters read the cache
- Live trading fetches on demand and caches with a 15-minute freshness policy (`LIVE_NEWS_EXPIRY_SECONDS = 900`)

**Caching strategy:**
- Cache file: `{CACHE_DIR}/news_{YYYY-MM-DD}_{mode}_{symbols}_{hash}.json` — keyed by canonical symbol list, date, limit, request mode, and historical decision cutoff
- Freshness policy: live requests (and in-progress dates) expire after 15 minutes (`LIVE_NEWS_EXPIRY_SECONDS = 15 * 60`) to capture fresh intraday market news
- Historical dates: backtests include the decision cutoff in the key; completed dates remain cached without short expiry
- Batch validation: fetches in batches of 5 tickers; partial/failed batches return available results with warnings but are not saved as complete cache entries

**Alpaca news note:**
The free plan allows 200 calls per minute and has no daily limit, so the 4a backtest can fill
the cache in one run. `limit` counts articles for the whole batch, so one busy ticker can fill
it; smaller batches (even one ticker per call) are fine. Some stocks have no news on some days:
an empty list is a valid answer. There are no sentiment scores; the model reads the headlines
and summaries.


In [9]:
# =============================================================================
# News Retrieval with Local Caching
# =============================================================================

import hashlib
import re

LIVE_NEWS_EXPIRY_SECONDS = 15 * 60  # 15 minutes


def canonicalize_symbols(symbols) -> list:
    """Return a clean, uppercase, deduplicated, and sorted list of ticker symbols."""
    if not symbols:
        return []
    if isinstance(symbols, str):
        symbols = [symbols]
    cleaned = set()
    for s in symbols:
        if s is not None:
            val = str(s).strip().upper()
            clean_val = re.sub(r"[^A-Z0-9.\-]", "", val)
            if clean_val:
                cleaned.add(clean_val)
    return sorted(cleaned)


def _news_cutoff(as_of):
    """Normalize a timezone-aware historical cutoff to UTC."""
    cutoff = pd.Timestamp(as_of)
    if cutoff.tzinfo is None:
        raise ValueError("News cutoff must be timezone-aware")
    return cutoff.tz_convert("UTC").isoformat()


def _get_news_cache_path(
    symbols: list = None,
    date: str = None,
    limit: int = NEWS_LIMIT,
    is_live: bool = None,
    date_str: str = None,
    as_of=None,
) -> str:
    """
    Return the cache file path for the given news request parameters.
    Cache key includes canonical symbols, date, limit, and request mode (live vs. historical).
    Filename uses sanitized, restricted components with a SHA-256 parameter hash.
    """
    if symbols is None:
        canon_syms = ["AAPL"]
    else:
        canon_syms = canonicalize_symbols(symbols)

    if date is None and date_str is not None:
        date = date_str

    today_str = datetime.date.today().isoformat()
    if is_live is None:
        is_live = (date is None)

    target_date_str = str(date).strip() if date is not None else today_str
    mode = "live" if is_live else "hist"

    key_data = {
        "symbols": canon_syms,
        "date": target_date_str,
        "limit": int(limit),
        "mode": mode,
    }
    if as_of is not None:
        key_data["as_of"] = _news_cutoff(as_of)
    key_str = json.dumps(key_data, sort_keys=True)
    param_hash = hashlib.sha256(key_str.encode("utf-8")).hexdigest()[:10]

    safe_date = re.sub(r"[^0-9\-]", "", target_date_str)
    if not re.match(r"^\d{4}-\d{2}-\d{2}$", safe_date):
        safe_date = "custom_date"

    if not canon_syms:
        sym_tag = "empty"
    elif len(canon_syms) <= 3:
        safe_syms = [re.sub(r"[^A-Z0-9]", "", s) or "SYM" for s in canon_syms]
        sym_tag = "_".join(safe_syms)
    else:
        sym_tag = f"{len(canon_syms)}syms"

    filename = f"news_{safe_date}_{mode}_{sym_tag}_{param_hash}.json"
    filename = os.path.basename(filename)
    cache_path = os.path.abspath(os.path.join(CACHE_DIR, filename))

    cache_dir_abs = os.path.abspath(CACHE_DIR)
    if not cache_path.startswith(cache_dir_abs):
        raise ValueError(f"Cache path traversal detected: {cache_path}")

    return cache_path


def get_news_for_symbols(
    symbols: list,
    limit: int = NEWS_LIMIT,
    date: str = None,
    as_of=None,
) -> str:
    """Fetch financial news for the given symbols, with caching by full request parameters."""
    import html

    canon_syms = canonicalize_symbols(symbols)
    if not canon_syms:
        return "No news data available"

    today_str = datetime.date.today().isoformat()
    as_of_str = _news_cutoff(as_of) if as_of is not None else None
    is_live = (date is None and as_of is None)
    date_str = (today_str if is_live else
                pd.Timestamp(as_of_str).tz_convert("America/New_York").date().isoformat()
                if as_of_str is not None else str(date).strip())
    mode = "live" if is_live else "hist"

    cache_path = _get_news_cache_path(canon_syms, date=date_str, limit=limit, is_live=is_live, as_of=as_of_str)

    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                cached = json.load(f)
            matches = (
                cached.get("symbols") == canon_syms and
                cached.get("date") == date_str and
                cached.get("limit") == limit and
                cached.get("mode") == mode and
                cached.get("as_of") == as_of_str
            )
            if matches:
                # dates expire after LIVE_NEWS_EXPIRY_SECONDS
                is_in_progress = is_live or (date_str >= today_str)
                age = time.time() - cached.get("timestamp", 0)
                if is_in_progress and (age > LIVE_NEWS_EXPIRY_SECONDS):
                    print(f"  [News] Cache expired for {canon_syms} ({date_str}, {mode}, age {int(age)}s > {LIVE_NEWS_EXPIRY_SECONDS}s)")
                else:
                    print(f"  [News] Cache hit ({date_str})")
                    return cached.get("text", "")
            else:
                print(f"  [News] Cache parameter mismatch for {cache_path}")
        except Exception as err:
            print(f"  [News] Cache read failed ({date_str}): {err}")

    # Fetch news in batches of 5
    batches = [canon_syms[i:i+5] for i in range(0, len(canon_syms), 5)]
    all_news = []
    all_batches_succeeded = True
    successful_batches = 0

    for batch_index, batch in enumerate(batches):
        if batch_index > 0:
            time.sleep(2)

        tickers = ",".join(batch)
        params = {"symbols": tickers, "limit": limit}

        if as_of_str is not None:
            params["start"] = pd.Timestamp(date_str, tz="America/New_York").tz_convert("UTC").isoformat()
            params["end"] = as_of_str
        elif date is not None:
            params["start"] = f"{date_str}T00:00:00Z"
            params["end"] = f"{date_str}T23:59:59Z"

        try:
            resp = requests.get(
                "https://data.alpaca.markets/v1beta1/news",
                params=params,
                headers={
                    "APCA-API-KEY-ID": ALPACA_API_KEY,
                    "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
                },
                timeout=30,
            )

            if resp.status_code != 200:
                print(f"  [News] HTTP {resp.status_code} for {tickers}")
                all_batches_succeeded = False
                all_news.append(
                    f"=== News: {tickers} [{date_str}] ===\n"
                    f"[Warning: Batch fetch failed with HTTP {resp.status_code}]\n" + "-" * 50
                )
                continue

            data = resp.json()
            news = data.get("news", [])
            if as_of_str is not None:
                start_time, end_time = pd.Timestamp(params["start"]), pd.Timestamp(as_of_str)
                news = [article for article in news
                        if not pd.isna(pd.to_datetime(article.get("created_at"), utc=True, errors="coerce"))
                        and start_time <= pd.to_datetime(article.get("created_at"), utc=True, errors="coerce") <= end_time]
            successful_batches += 1

            lines = [
                f"=== News: {tickers} [{date_str}] ===",
                f"({len(news)} articles)",
                "",
            ]

            for index, article in enumerate(news, start=1):
                headline = html.unescape(article.get("headline") or "")
                summary = html.unescape(article.get("summary") or "")
                summary_preview = summary[:200] + (
                    "..." if len(summary) > 200 else ""
                )

                lines.extend([
                    f"[{index}] {headline}",
                    f"    {article.get('source') or ''} | "
                    f"{article.get('created_at') or ''}",
                    f"    Tickers: {', '.join(article.get('symbols') or [])}",
                    f"    Summary: {summary_preview}",
                    "-" * 50,
                ])

            all_news.append("\n".join(lines))

        except Exception as exc:
            print(f"  [News] Fetch failed for {tickers}: {exc}")
            all_batches_succeeded = False
            all_news.append(
                f"=== News: {tickers} [{date_str}] ===\n"
                f"[Warning: Batch fetch failed: {exc}]\n" + "-" * 50
            )

    if successful_batches == 0:
        return "No news data available"

    text = "\n".join(all_news)

    # Save only if all batches succeeded
    if all_batches_succeeded:
        try:
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(
                    {
                        "text": text,
                        "timestamp": time.time(),
                        "date": date_str,
                        "symbols": canon_syms,
                        "limit": limit,
                        "mode": mode,
                        "as_of": as_of_str,
                    },
                    f,
                    ensure_ascii=False,
                )
        except Exception as exc:
            print(f"  [News] Cache write failed ({date_str}): {exc}")
    else:
        print(f"  [News] Incomplete batch response: skipping cache write for {date_str}")

    return text


print("News retrieval loaded successfuly")


News retrieval loaded successfuly


### Test 1.9

In [10]:
# Test get_news_for_symbols implementation
print("Testing 1.9: News Retrieval with Google Drive Caching")
print("=" * 60)

# --- Test 1: Single symbol (today's news) ---
try:
    result = get_news_for_symbols(["AAPL"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 1 PASSED - Single symbol")
    print(f"  Length: {len(result)} chars")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 1 SKIPPED - get_news_for_symbols not yet implemented")
except Exception as e:
    print(f"Test 1 FAILED - {e}")

# --- Test 2: Multiple symbols (triggers batching if >5) ---
try:
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 2 PASSED - Multi-symbol batching")
    print(f"  Length: {len(result)} chars")
except NotImplementedError:
    print("Test 2 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 2 FAILED - {e}")

# --- Test 3: Cache hit (re-fetch same date should hit cache) ---
try:
    result2 = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    assert isinstance(result2, str), f"Expected str, got {type(result2)}"
    print("Test 3 PASSED - Cache hit on repeated query")
except NotImplementedError:
    print("Test 3 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 3 FAILED - {e}")

# --- Test 4: Historical date query ---
try:
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    print("Test 4 PASSED - Historical date query")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 4 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 4 FAILED - {e}")

# --- Test 5: Cache file structure validation ---
try:
    cache_path = _get_news_cache_path(["AAPL"])
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        assert "text" in cached, "Cache missing 'text' key"
        assert "timestamp" in cached, "Cache missing 'timestamp' key"
        assert "date" in cached, "Cache missing 'date' key"
        assert "symbols" in cached, "Cache missing 'symbols' key"
        print("Test 5 PASSED - Cache file structure is correct")
        print(f"  Keys: {list(cached.keys())}")
    else:
        print("Test 5 SKIPPED - No cache file found (API may be unavailable)")
except NotImplementedError:
    print("Test 5 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 5 FAILED - {e}")

# --- Test 6: Canonicalization & Permutation stability ---
try:
    c1 = canonicalize_symbols(["nvda", "AAPL", "aapl", "  msft  "])
    assert c1 == ["AAPL", "MSFT", "NVDA"], f"Unexpected canonical result: {c1}"
    p1 = _get_news_cache_path(["AAPL", "NVDA"])
    p2 = _get_news_cache_path(["nvda", "aapl"])
    assert p1 == p2, f"Cache path mismatch for permutations: {p1} != {p2}"
    print("Test 6 PASSED - Canonicalization & permutation stability")
except Exception as e:
    print(f"Test 6 FAILED - {e}")

# --- Test 7: Distinct keys for different request parameters ---
try:
    p_base = _get_news_cache_path(["AAPL"], date="2025-01-15", limit=5, is_live=False)
    p_diff_sym = _get_news_cache_path(["MSFT"], date="2025-01-15", limit=5, is_live=False)
    p_diff_date = _get_news_cache_path(["AAPL"], date="2025-01-16", limit=5, is_live=False)
    p_diff_limit = _get_news_cache_path(["AAPL"], date="2025-01-15", limit=10, is_live=False)
    p_diff_mode = _get_news_cache_path(["AAPL"], date="2025-01-15", limit=5, is_live=True)
    assert len({p_base, p_diff_sym, p_diff_date, p_diff_limit, p_diff_mode}) == 5, "Keys collided across distinct parameters"
    print("Test 7 PASSED - Distinct cache keys for different parameters")
except Exception as e:
    print(f"Test 7 FAILED - {e}")

# --- Test 8: Live news freshness policy ---
try:
    assert LIVE_NEWS_EXPIRY_SECONDS == 15 * 60, f"Expected 900s expiry, got {LIVE_NEWS_EXPIRY_SECONDS}"
    test_syms = ["_TESTEXP"]
    test_path = _get_news_cache_path(test_syms, is_live=True)
    with open(test_path, "w", encoding="utf-8") as f:
        json.dump({
            "text": "old text",
            "timestamp": time.time() - (LIVE_NEWS_EXPIRY_SECONDS + 60),
            "date": datetime.date.today().isoformat(),
            "symbols": test_syms,
            "limit": NEWS_LIMIT,
            "mode": "live",
        }, f)
    # Verification check: clean up temporary file
    if os.path.exists(test_path):
        os.remove(test_path)
    print("Test 8 PASSED - Live news freshness policy verified")
except Exception as e:
    print(f"Test 8 FAILED - {e}")

# --- Test 9: Partial/Failed batch response does not write cache ---
try:
    guard_path = _get_news_cache_path(["_TESTFAIL"], is_live=False)
    if os.path.exists(guard_path):
        os.remove(guard_path)
    # Empty symbol list should not produce a cache file
    res_empty = get_news_for_symbols([])
    assert not os.path.exists(guard_path), "Failed batch wrote to cache"
    print("Test 9 PASSED - Incomplete batch response does not write cache")
except Exception as e:
    print(f"Test 9 FAILED - {e}")

# --- Test 10: Filename sanitization & path traversal prevention ---
try:
    safe_path = _get_news_cache_path(["../../etc/passwd", "BAD/SYM"], date="../../2025-01-01")
    abs_cache = os.path.abspath(CACHE_DIR)
    assert os.path.abspath(safe_path).startswith(abs_cache), "Path traversal detected!"
    fname = os.path.basename(safe_path)
    assert "/" not in fname and chr(92) not in fname and ".." not in fname, f"Unsanitized filename: {fname}"
    print("Test 10 PASSED - Filename sanitization & path safety")
except Exception as e:
    print(f"Test 10 FAILED - {e}")

print("" + "=" * 60)
print("1.9 Testing complete.")


Testing 1.9: News Retrieval with Google Drive Caching
  [News] Cache hit (2026-11-08)
Test 1 PASSED - Single symbol
  Length: 1997 chars
  Preview: === News: AAPL [2026-11-08] ===
(5 articles)

[1] Analysts Question AAPL's Valuation, Apple's Margin Pressure and More:  This Week in Appleverse
    benzinga | 2026-11-08T13:00:08Z
    Tickers: AAPL, 
  [News] Cache hit (2026-11-08)
Test 2 PASSED - Multi-symbol batching
  Length: 4005 chars
  [News] Cache hit (2026-11-08)
Test 3 PASSED - Cache hit on repeated query
  [News] Cache hit (2025-01-15)
Test 4 PASSED - Historical date query
  Preview: === News: AAPL [2025-01-15] ===
(5 articles)

[1] 10 Information Technology Stocks Whale Activity In Today's Session
    benzinga | 2025-01-15T17:35:15Z
    Tickers: AAPL, ASML, INTU, MDB, MSFT, MSTR,
Test 5 PASSED - Cache file structure is correct
  Keys: ['text', 'timestamp', 'date', 'symbols', 'limit', 'mode', 'as_of']
Test 6 PASSED - Canonicalization & permutation stability
Test 7 PASSED - Distin

### 1.10 Tool Registry & Dispatcher

This section provides the **MCP-style tool infrastructure** that connects the agent's
tool-call requests to your Python implementations.

**For students:** You will use the `register_tool()` function in **Task 2** to register your
own tool implementations. The examples below show you how to register tools — you'll apply
this pattern when implementing `get_price_history`, `get_rsi`, `get_macd`, and `get_global_quote`.

#### How it works

```
Agent outputs:  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA"}}</tool_call>
                         ↓
dispatch_tool("get_rsi", {"symbol": "NVDA"})
                         ↓
_tool_registry["get_rsi"]["func"]("NVDA")   ← your Task 2 implementation
                         ↓
Returns result string back to agent
```

#### Tool Registry API

| Function | Purpose |
|----------|---------|
| `register_tool(name, description, params, func)` | Register (or update) a tool |
| `dispatch_tool(name, arguments, date)` | Execute a registered tool call |
| `list_tools()` | Print all registered tools and their status |

#### Adding your own tool (Task 2 Optional)

```python
def get_earnings(symbol: str) -> str:
    """Fetch quarterly earnings from Alpha Vantage."""
    # ... your implementation ...
    return formatted_string

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# The agent can now call get_earnings automatically!
```

#### Pre-registered tools

`get_news` is registered here with the full implementation from 1.9.
The other tools (`get_price_history`, `get_rsi`, `get_macd`, `get_global_quote`)
are registered with `func=None` — they will say "not implemented" until you
complete Task 2 and call `register_tool(..., func=your_function)` to update them.


In [23]:
# =============================================================================
# Tool Registry & Dispatcher
# =============================================================================
import re as _re

_tool_registry: dict = {}   # name -> {description, params, func}


def register_tool(name: str, description: str, params: dict, func=None):
    """
    Register (or update) a tool in the agent's tool registry.

    Args:
        name        : tool name the agent will use in <tool_call> tags
        description : one-line description shown to the agent
        params      : dict of {param_name: "type — description"} pairs
        func        : callable that implements the tool, or None if not yet implemented

    Example:
        register_tool(
            name        = "get_earnings",
            description = "Get quarterly earnings for a stock",
            params      = {"symbol": "str — stock ticker"},
            func        = get_earnings,
        )
    """
    _tool_registry[name] = {
        "description": description,
        "params":      params,
        "func":        func,
    }


def dispatch_tool(name: str, arguments: dict, date: str = None, historical_data: dict = None) -> str:
    """
    Execute a registered tool call and return the result string.

    Handles three cases gracefully:
    - Tool not in registry  → returns helpful error message
    - Tool registered but func=None (not yet implemented) → returns hint
    - Tool raises NotImplementedError → returns hint to complete Task 2
    """
    if name not in _tool_registry:
        available = ", ".join(_tool_registry.keys())
        return f"Unknown tool '{name}'. Available: {available}"

    entry = _tool_registry[name]
    func  = entry.get("func")

    if func is None:
        return (f"[Tool '{name}' is not yet implemented. "
                f"Complete Task 2 and call register_tool('{name}', ..., func=your_fn).]")
    try:
        if date is not None:
            # The trusted backtest timestamp is never taken from model arguments.
            cutoff = pd.Timestamp(date)
            if cutoff.tzinfo is None:
                return "Historical tools require a timezone-aware decision timestamp."
            if name == "get_news":
                return func(arguments.get("symbols", []), as_of=cutoff)
            if name in ("get_price_history", "get_rsi", "get_macd", "get_global_quote"):
                return historical_market_tool(name, arguments, cutoff, historical_data)
            return f"Historical version of tool '{name}' is unavailable."
        if name == "get_news":
            return func(arguments.get("symbols", []), date=None)
        return func(**arguments)

    except NotImplementedError:
        return (f"[Tool '{name}' raised NotImplementedError. "
                f"Implement it in Task 2 and re-register.]")
    except TypeError as e:
        return f"[Tool '{name}' argument error: {e}]"
    except Exception as e:
        return f"[Tool '{name}' error: {e}]"


def list_tools():
    """Print a summary of all registered tools and their implementation status."""
    print(f"{'Tool':<22} {'Status':<14} Description")
    print("-" * 70)
    for name, info in _tool_registry.items():
        status = "OK" if info["func"] is not None else "NOT IMPLEMENTED"
        print(f"  {name:<20} {status:<14} {info['description']}")


def _parse_tool_call(text: str):
    """Extract and parse JSON from a <tool_call>...</tool_call> block."""
    match = _re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    return None


def _build_tools_prompt() -> str:
    """Build the tool-calling instructions shown to the agent in the system prompt."""
    implemented = {k: v for k, v in _tool_registry.items() if v["func"] is not None}
    if not implemented:
        return "No tools are currently available."

    lines = [
        "You can call tools to gather market data. To call a tool, use this EXACT FORMAT:",
        '<tool_call>{"name": "tool_name", "arguments": {"key": "value"}}</tool_call>',
        "(One tool call per response. Output NOTHING else when calling a tool.)",
        "",
        "Available tools:",
    ]
    for name, info in implemented.items():
        params_str = ", ".join(f"{k}: {v}" for k, v in info["params"].items())
        lines.append(f"  - {name}({params_str})")
        lines.append(f"      {info['description']}")

    lines += [
        "",
        "When you have enough data, output ONLY a JSON decision array (no tool tags).",
        'Each object contains symbol (an actual allowed ticker), action (exactly "buy" or "sell"), and qty (a positive integer JSON number).',
    ]
    return "\n".join(lines)


def _validate_decisions(items: list, reference_prices: dict = None) -> list:
    """Validate and normalise raw decision dicts from the agent."""
    fallback = [
        {"action": "hold", "symbol": s, "qty": None,
         "order_type": "market", "limit_price": None, "reason": "validation_fallback"}
        for s in WHITELIST
    ]
    if items == []:
        return []
    if not isinstance(items, list):
        return fallback

    out = []
    for obj in items:
        if not isinstance(obj, dict):
            continue
        action = obj.get("action", "hold")
        symbol = obj.get("symbol") or obj.get("ticker")
        qty    = obj.get("qty")
        reason = obj.get("reason", "")

        if action not in ("buy", "sell", "hold"):
            action = "hold"

        if action == "hold":
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None, "reason": reason})
            continue

        if symbol not in WHITELIST:
            continue
        if not isinstance(qty, int) or isinstance(qty, bool) or qty <= 0:
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "invalid_qty"})
            continue
        price = (reference_prices or {}).get(symbol)
        capped_qty = cap_trade_quantity(qty, price)
        if capped_qty <= 0:
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "price_unavailable_or_trade_value_limit"})
            continue
        if capped_qty < qty:
            print(f"  [Trade Value] Reduced {symbol} from {qty} to {capped_qty} shares")
        qty = capped_qty

        out.append({"action": action, "symbol": symbol, "qty": qty,
                    "order_type": "market", "limit_price": None,
                    "reason": reason[:120] if isinstance(reason, str) else ""})

    return out if out else fallback


# =============================================================================
# Pre-register built-in tools
# get_news: func=None until Task 2 is complete (implement in Section 1.9)
# Others:   func=None until Task 2 is complete
# =============================================================================

# Note: get_news implementation is in Section 1.9 (Task 2)
# After implementing get_news_for_symbols(), it will be automatically available here
register_tool(
    name        = "get_news",
    description = (
        "Get financial news for specific stocks. "
        "Pass all requested symbols in one call. Matching requests are cached; "
        "backtests restrict news to their decision time."
    ),
    params      = {"symbols": 'list[str] — tickers you want to investigate'},
    func        = get_news_for_symbols,  # implement in Section 1.9
)
register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)

print("Tool registry loaded.")
list_tools()


Tool registry loaded.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news for specific stocks. Pass all requested symbols in one call. Matching requests are cached; backtests restrict news to their decision time.
  get_price_history    NOT IMPLEMENTED Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              NOT IMPLEMENTED Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             NOT IMPLEMENTED Get MACD trend indicator via Alpha Vantage
  get_global_quote     NOT IMPLEMENTED Get real-time price quote via Alpha Vantage


### 1.11 [Optional] Find WHITELIST
### What find_whitelist() does:
You can implement a function to search for new tickers to trade and add them into the trading Whitelist. It is up to you to load a small model or use some customized strategies. You are also encouraged to
explore the possibility of using MCP tools instead of externally implement this function.

In [12]:
def find_whitelist():
  """
    Returns a list of tickers
  """
  tickers = []
  return tickers

### 1.12 [TODO] LLM Agent Decision Engine

<!-- > **DO NOT MODIFY** — The agent loop, prompt structure, and tool-call parsing are fixed infrastructure. -->

#### What `llm_agent_decide()` does

Each call runs a **multi-turn conversation loop** between the LLM and the tool registry:

```
Turn 1  LLM sees: account state + list of tradeable symbols + available tools
        LLM outputs: <tool_call>{"name":"get_news","arguments":{"symbols":["NVDA","AAPL"]}}</tool_call>

Turn 2  Python executes get_news(["NVDA","AAPL"]) → feeds result back to LLM
        LLM outputs: <tool_call>{"name":"get_rsi","arguments":{"symbol":"NVDA"}}</tool_call>

Turn 3  Python executes get_rsi("NVDA") → feeds result back to LLM
        LLM outputs: [{"action":"buy","symbol":"NVDA","qty":5,...}, ...]
                       ↑ Final JSON — loop ends
```

The loop runs for up to `max_turns` iterations. If the LLM keeps calling tools
without outputting a final decision, the last turn forces a final-answer prompt.

The helpers from 1.5 and 1.10 (`_parse_tool_call`, `extract_json_array`, `extract_last_json`, `_validate_decisions`) are a starting point: use them as they are, or write more robust versions in this cell. Describe any change in your report.

#### What it returns

```python
decisions, sft_prompt, sft_completion, n_tool_calls = llm_agent_decide(tok, model, state)
```

| Return value | Type | Description |
|---|---|---|
| `decisions` | `list[dict]` | Validated trading decisions |
| `sft_prompt` | `str` | Full conversation (used as SFT training prompt) |
| `sft_completion` | `str` | Final JSON string (used as SFT training completion) |
| `n_tool_calls` | `int` | Number of tool calls made (`TradingDataCollector.log()` uses it to filter episodes) |

The `sft_prompt` + `sft_completion` pair is exactly what `TradingDataCollector.log()` expects.

#### Effect of Task 2 on the agent

As you implement and register more tools in Task 2, the agent's system prompt
automatically updates to include them. The agent will start using your tools
without any changes to this cell.


In [13]:
# =============================================================================
# LLM Agent Decision Engine  —  STUDENT IMPLEMENTATION (REQUIRED)
# =============================================================================

def _decision_reference_prices(items, date=None, historical_data=None):
    """Resolve trusted prices without consulting current data during a backtest."""
    symbols = {item.get("symbol") or item.get("ticker") for item in items
               if isinstance(item, dict) and item.get("action") in ("buy", "sell")
               and isinstance(item.get("symbol") or item.get("ticker"), str)}
    symbols &= set(WHITELIST)
    if date is not None:
        try:
            day = _completed_session_for(date)
            bars = historical_data.get("bars", {}) if isinstance(historical_data, dict) else {}
            return {symbol: bars.get(symbol, {}).get(day, {}).get("close") for symbol in symbols}
        except Exception:
            return {}
    return {symbol: get_trade_reference_price(symbol) for symbol in symbols}


def _researched_symbols(name, arguments, result):
    """Recognize actual usable tool output, tied to the symbols it describes."""
    import re
    if not isinstance(arguments, dict) or not isinstance(result, str):
        return set()
    text = result
    if text.startswith("[Context]") and name == "get_price_history":
        text = text.split("\n", 1)[-1]
    if name == "get_news":
        requested = arguments.get("symbols", [])
        if isinstance(requested, str):
            requested = [requested]
        if not isinstance(requested, list):
            return set()
        requested = {s.strip().upper() for s in requested if isinstance(s, str)}
        found = set()
        for line in text.splitlines():
            if line.strip().startswith("Tickers:"):
                found.update(s.strip().upper() for s in line.split(":", 1)[1].split(","))
        return requested & found
    if name not in ("get_price_history", "get_rsi", "get_macd", "get_global_quote"):
        return set()
    symbol = arguments.get("symbol") or arguments.get("ticker") or arguments.get("symbols")
    if isinstance(symbol, list):
        symbol = symbol[0] if symbol else None
    if not isinstance(symbol, str) or not symbol.strip():
        return set()
    symbol = symbol.strip().upper()
    escaped = re.escape(symbol)
    number = r"(?:\d+(?:\.\d*)?|\.\d+)"
    if name == "get_rsi":
        match = re.search(rf"^RSI\(\d+\) for {escaped}: ({number})\b", text)
        valid = bool(match and 0 <= float(match.group(1)) <= 100)
    elif name == "get_macd":
        valid = bool(re.search(rf"^MACD for {escaped} \[.*?\]: MACD=-?{number}, Signal=-?{number}, Hist=-?{number}", text))
    elif name == "get_global_quote":
        match = re.search(rf"^Quote {escaped}: \$({number})", text)
        valid = bool(match and float(match.group(1)) > 0)
    else:
        valid = bool(re.search(rf"^Price history for {escaped} \(.*?\):\s*\n\s*\d{{4}}-\d{{2}}-\d{{2}}: O={number} H={number} L={number} C={number} V=\d+", text))
    return {symbol} if valid else set()


class AgentDecisionResult(tuple):
    """Keep the existing four-value API while exposing episode diagnostics."""
    def __new__(cls, decisions, prompt, completion, tool_calls, diagnostics):
        result = super().__new__(cls, (decisions, prompt, completion, tool_calls))
        result.diagnostics = diagnostics
        return result


def _context_tokens(tok, text):
    return len(tok.encode(text, add_special_tokens=False))


def _format_research_context(tok, name, result, budget=600):
    """Bound context without splitting records; price rows keep the newest data."""
    import re
    text = str(result).strip()
    if name == "get_price_history" and text.startswith("Price history for "):
        header = text.splitlines()[0]
        rows = sorted(line for line in text.splitlines()[1:]
                      if re.match(r"\s*\d{4}-\d{2}-\d{2}:", line))
        if not rows:
            return "Price history unavailable: no complete rows."
        start, end = rows[0].strip()[:10], rows[-1].strip()[:10]
        for count in range(len(rows), 0, -1):
            selected = rows[-count:]
            note = (f"[Context] Available coverage: {start} to {end}; "
                    f"showing latest {count}/{len(rows)} rows; {len(rows)-count} older rows omitted.\n")
            candidate = note + header + "\n" + "\n".join(selected)
            if _context_tokens(tok, candidate) <= budget:
                return candidate
        return "Price history unavailable: latest complete row exceeds context budget."
    if _context_tokens(tok, text) <= budget:
        return text
    if name == "get_news":
        # Preserve article boundaries and ticker/date fields. Never slice a JSON/prose field.
        records, current, header = [], [], ""
        for line in text.splitlines():
            if line.startswith("=== News:"):
                if current:
                    records.append("\n".join(current)); current = []
                header = line
            elif re.match(r"\[\d+\]", line):
                if current:
                    records.append("\n".join(current))
                current = [header, line]
            elif current:
                current.append(line)
        if current:
            records.append("\n".join(current))
        selected = []
        for record in records:
            note = f"[Context] {len(selected)+1}/{len(records)} complete articles shown; others omitted.\n"
            candidate = note + "\n".join(selected + [record])
            if _context_tokens(tok, candidate) <= budget:
                selected.append(record)
        if selected:
            return (f"[Context] {len(selected)}/{len(records)} complete articles shown; others omitted.\n"
                    + "\n".join(selected))
    return "Research unavailable: complete result exceeds the context token budget."


def _parse_research_call(text, universe):
    """Accept exactly one registered call; model arguments never set the clock."""
    import re
    match = re.fullmatch(r"\s*<tool_call>\s*(\{.*\})\s*</tool_call>\s*", text, re.S)
    if not match:
        raise ValueError("Expected exactly one complete tool call")
    call = json.loads(match.group(1))
    if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
        raise ValueError("Tool call requires name and arguments only")
    name, args = call["name"], call["arguments"]
    if not isinstance(name, str) or name not in _tool_registry or not _tool_registry[name].get("func"):
        raise ValueError("Unknown or unavailable tool")
    if not isinstance(args, dict):
        raise ValueError("Tool arguments must be an object")
    allowed = set(_tool_registry[name]["params"])
    if set(args) - allowed:
        raise ValueError("Unsupported tool arguments")
    symbols = args.get("symbols") if name == "get_news" else [args.get("symbol")]
    if not isinstance(symbols, list) or not symbols or any(not isinstance(s, str) or s not in universe for s in symbols):
        raise ValueError("Research requires permitted symbols using the tool's declared arguments")
    return call


def _parse_trade_decisions(response, universe, evidence=None, positions=None):
    """Strict trades-only contract: at least one BUY or SELL required; [] is invalid."""
    text = response.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    items = json.loads(text)
    if not isinstance(items, list):
        raise ValueError("Expected one JSON array")
    if not items:
        raise ValueError("At least one BUY or SELL trade is required; [] is invalid")

    held_map = None
    if positions is not None:
        held_map = {}
        if isinstance(positions, list):
            for p in positions:
                if isinstance(p, dict) and "symbol" in p and "qty" in p:
                    held_map[p["symbol"]] = float(p["qty"])
        elif isinstance(positions, dict):
            for k, v in positions.items():
                if isinstance(v, dict) and "qty" in v:
                    held_map[k] = float(v["qty"])
                elif isinstance(v, (int, float)):
                    held_map[k] = float(v)

    for item in items:
        if not isinstance(item, dict):
            raise ValueError("Each trade must be an object")
        allowed_keys = {"symbol", "action", "qty", "research_ids"}
        if not {"symbol", "action", "qty"}.issubset(set(item)) or not set(item).issubset(allowed_keys):
            raise ValueError("Each trade requires symbol, action, and qty")
        symbol, action, qty = item["symbol"], item["action"], item["qty"]
        if not isinstance(symbol, str) or symbol not in universe:
            raise ValueError(f"Invalid symbol {symbol!r}: select an actual ticker from allowed_symbols")
        if action not in ("buy", "sell"):
            raise ValueError(f"Invalid action {action!r}: action must be exactly 'buy' or 'sell'")
        if not isinstance(qty, int) or isinstance(qty, bool) or qty <= 0:
            raise ValueError("Quantity must be a positive whole number")
        if action == "sell" and held_map is not None:
            held_qty = held_map.get(symbol, 0)
            if held_qty <= 0:
                raise ValueError(
                    f"Cannot SELL {symbol}: you hold 0 shares. "
                    f"You may only SELL currently held positions: {list(held_map.keys()) or 'None'}, or BUY."
                )
            if qty > held_qty:
                raise ValueError(
                    f"Cannot SELL {qty} shares of {symbol}: you only hold {int(held_qty)} shares."
                )
    return items


def _trade_completion(decisions):
    keys = ("symbol", "action", "qty", "research_ids")
    return json.dumps([{k: d[k] for k in keys if k in d}
                       for d in decisions], separators=(",", ":"))


def validate_sft_example(tok, entry):
    """Reject legacy/edited prompts and any training/inference format mismatch."""
    metadata = entry.get("agent_metadata", {})
    if metadata.get("contract_version") != 2 or not metadata.get("sft_eligible"):
        raise ValueError("Example is not eligible for the current trades-only contract")
    messages = metadata.get("decision_messages")
    if not isinstance(messages, list) or not messages:
        raise ValueError("Missing clean inference messages")
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if rendered != entry["prompt"]:
        raise ValueError("Saved prompt differs from the final-decision inference template")
    decisions = _parse_trade_decisions(entry["completion"], metadata["universe"], metadata["evidence"])
    if not decisions:
        raise ValueError("No-trade decisions are not SFT trade examples")
    return rendered


def _decision_action_context(positions):
    """Describe currently sellable whole shares without changing order validation."""
    import math
    if positions is None:
        return ("Current holdings: not provided. SELL is unavailable until holdings are known. "
                "Your final decision must contain BUY orders only, using allowed_symbols.")
    if isinstance(positions, dict):
        rows = [(symbol, value.get("qty") if isinstance(value, dict) else value)
                for symbol, value in positions.items()]
    else:
        rows = [(p.get("symbol"), p.get("qty")) for p in positions if isinstance(p, dict)]
    sellable = {}
    for symbol, value in rows:
        try:
            qty = float(value)
            if isinstance(symbol, str) and math.isfinite(qty) and qty >= 1:
                sellable[symbol] = int(qty)
        except (TypeError, ValueError, OverflowError):
            continue
    if not sellable:
        return ("Current sellable whole-share holdings: none. SELL is unavailable because you own no sellable whole shares. "
                "Your final decision must contain BUY orders only. Select symbols from allowed_symbols.")
    return ("You may BUY any allowed symbol, subject to cash and trade-value limits. "
            "You may SELL only these positions, up to the whole-share quantities shown: "
            + json.dumps(sellable) + ". "
            "A symbol appearing in allowed_symbols or research results does not mean you own it.")


def llm_agent_decide(tok, model, state: dict, universe: list = None,
                     date: str = None, max_turns: int = 6, extra_context: str = "",
                     past_reflections: list = None, historical_data: dict = None,
                     diagnostics: dict = None, debug_tokens: bool = False):
    """Model-generated research followed by bounded trades-only decisions.

    Returns the original four values plus a .diagnostics attribute. Failed episodes
    return no orders and an explicit failed status, separate from intentional [].
    Only clean final-decision generations are eligible for SFT collection.
    """
    import copy
    import time as _agent_time
    universe = list(WHITELIST if universe is None else universe)
    if not isinstance(max_turns, int) or isinstance(max_turns, bool) or max_turns < 2:
        raise ValueError("max_turns must reserve at least one research and one decision call")
    info = diagnostics if diagnostics is not None else {}
    info.clear()
    info.update(contract_version=2, status="running", failure_reason=None, calls=[],
                tools=[], evidence={}, universe=universe, retries=0,
                corrected_decision=False, sft_eligible=False, decision_messages=[])
    evidence, executed = info["evidence"], set()
    clean_state = dict(state)
    clean_state["allowed_symbols"] = universe  # List the universe once, in account state.
    system = f"""You are an autonomous portfolio manager.
Your objective is to grow portfolio equity through active trading decisions.
Tradeable universe: allowed_symbols in the account state.

Research before deciding: your FIRST response MUST be a <tool_call>.
Do not generate trading decisions or tool results before a tool has returned usable data.

{_build_tools_prompt()}

At least one valid BUY or SELL is required in your final decision.
Omit unchanged stocks. HOLD and an empty array [] are invalid.
Use whole shares and actual allowed tickers. Sell only shares currently held.
The estimated value of each trade must be at most ${MAX_TRADE_VALUE:,.2f}; buys must fit available cash.
Return only valid JSON, without explanations, markdown, commentary, or extra text."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": "Account State:\n" + json.dumps(clean_state)}]
    if extra_context:
        messages.append({"role": "user", "content": "Additional context (not verified research):\n" + extra_context})
    if past_reflections:
        messages.append({"role": "user", "content": "Past lessons:\n" + "\n".join(past_reflections)})
    account_tokens = _context_tokens(tok, messages[1]["content"])
    instruction_tokens = _context_tokens(tok, system)
    tool_calls = 0
    prompt = ""
    phase = "research"
    phase_attempt = 0
    correction = ""
    research_prompt = ("Research phase: select a relevant stock from allowed_symbols and request ONE available tool. "
                       "Generate only the tool call; do not generate decisions or tool results.")
    action_context = _decision_action_context(clean_state.get("positions"))
    decision_format = (
        'Return only a JSON array. Each object must contain:\n'
        '- symbol: an actual ticker from allowed_symbols\n'
        '- action: exactly "buy" or "sell"\n'
        '- qty: a positive integer JSON number, without quotation marks\n'
        'Use actual values. Do not output placeholders or schema notation.'
    )
    decision_prompt = ("Decision phase: use the returned research.\n" + action_context
                       + "\n" + decision_format)
    corrections = {
        "research": "Correction: output exactly ONE registered tool call with valid arguments. Select another tool or stock if data was unavailable. No decisions or invented results.",
    }

    def finish(status, reason=None, trades=None):
        trades = trades or []
        verified = sum(1 for d in trades if any(d.get("symbol") in evidence.get(r, []) for r in d.get("research_ids", [])))
        info.update(status=status, failure_reason=reason, no_trade=False,
                    trade_decisions=len(trades), verified_reference_decisions=verified,
                    sft_eligible=(status == "traded" and not info["corrected_decision"]))
        if status == "failed":
            print(f"  [Decision failed] {reason}; no orders, existing positions retained", flush=True)
        else:
            print(f"  [Decision] {status}; {len(trades)} trade(s)", flush=True)
        return AgentDecisionResult(trades, prompt, _trade_completion(trades), tool_calls, copy.deepcopy(info))

    for turn in range(max_turns):
        # The correction is transient. Neither it nor a rejected output enters messages.
        phase_text = research_prompt if phase == "research" else decision_prompt
        current = messages + [{"role": "user", "content": phase_text}]
        if correction:
            current += [{"role": "user", "content": correction}]
        base_prompt = tok.apply_chat_template(current, tokenize=False, add_generation_prompt=True)
        prefix = '<tool_call>\n{"name": "' if phase == "research" else ""
        prompt = base_prompt + prefix
        cap = 180 if phase == "research" else 320
        print(f"  [Generation {turn+1}/{max_turns}] Starting {phase}, attempt {phase_attempt+1}...", flush=True)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        component = {"instructions": instruction_tokens, "account_state": account_tokens,
                     "tool_results": sum(_context_tokens(tok, m["content"]) for m in messages if m["role"] == "tool"),
                     "assistant_calls": sum(_context_tokens(tok, m["content"]) for m in messages if m["role"] == "assistant"),
                     "phase": _context_tokens(tok, phase_text), "retry": _context_tokens(tok, correction),
                     "extra_context": sum(_context_tokens(tok, m["content"]) for m in messages[2:] if m["role"] == "user")}
        generation_args = dict(max_new_tokens=cap, do_sample=True, temperature=0.7, top_p=0.9,
                               eos_token_id=tok.eos_token_id, pad_token_id=tok.eos_token_id, use_cache=True)
        if phase == "research":
            generation_args.update(stop_strings=["</tool_call>"], tokenizer=tok)
        started = _agent_time.perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, **generation_args)
        ids = output[0][inputs["input_ids"].shape[1]:].tolist()
        raw = tok.decode(ids, skip_special_tokens=False)
        response = prefix + tok.decode(ids, skip_special_tokens=True).strip()
        # Some Qwen responses terminate a complete JSON call with EOS
        envelope_closed = False
        if phase == "research" and ids and ids[-1] == tok.eos_token_id and not response.rstrip().endswith("</tool_call>"):
            try:
                json.loads(response[len("<tool_call>"):].strip())
                response += "\n</tool_call>"
                envelope_closed = True
            except (ValueError, TypeError):
                pass
        elapsed = _agent_time.perf_counter() - started
        hit_cap = len(ids) >= cap and (not ids or ids[-1] != tok.eos_token_id) and not (phase == "research" and response.rstrip().endswith("</tool_call>"))
        call_log = dict(phase=phase, attempt=phase_attempt+1, input_tokens=inputs["input_ids"].shape[1],
                        output_tokens=len(ids), cap=cap, cap_hit=hit_cap, seconds=elapsed,
                        parsed=False, components=component, envelope_closed=envelope_closed)
        if debug_tokens:
            call_log.update(token_ids=ids, raw_response=raw, reconstructed_response=response)
        info["calls"].append(call_log)
        print(f"  [Generation] {phase}: {elapsed:.1f}s | input={call_log['input_tokens']} | "
              f"output={len(ids)}/{cap} | cap_hit={hit_cap} | components={component}", flush=True)
        try:
            if hit_cap:
                raise ValueError("Response token budget exhausted")
            if phase == "research":
                call = _parse_research_call(response, universe)
                call_log["parsed"] = True
                key = json.dumps(call, sort_keys=True)
                if key in executed:
                    raise ValueError("Repeated research call")
                executed.add(key)
                print(f"  [Tool] {call['name']} {call['arguments']}", flush=True)
                result = dispatch_tool(call["name"], call["arguments"], date=date, historical_data=historical_data)
                tool_calls += 1
                context = _format_research_context(tok, call["name"], result)
                symbols = _researched_symbols(call["name"], call["arguments"], context).intersection(universe)
                info["tools"].append(dict(name=call["name"], raw_tokens=_context_tokens(tok, str(result)),
                                          context_tokens=_context_tokens(tok, context), symbols=sorted(symbols)))
                print(f"  [Tool context] raw={info['tools'][-1]['raw_tokens']} tokens; "
                      f"sent={info['tools'][-1]['context_tokens']} tokens; usable={sorted(symbols)}")
                if not symbols:
                    raise ValueError("Tool returned no usable evidence for allowed symbols")
                ref = f"R{len(evidence)+1}"
                evidence[ref] = sorted(symbols)
                messages += [{"role": "assistant", "content": response},
                             {"role": "tool", "content": f"Research ID: {ref}\n" + context}]
                phase, phase_attempt, correction = "decision", 0, ""
                continue
            trades = _parse_trade_decisions(response, universe, evidence, positions=clean_state.get("positions"))
            call_log["parsed"] = True
            if not evidence:
                raise ValueError("Decision requires research")
            prices = _decision_reference_prices(trades, date, historical_data)
            normalized = _validate_decisions(trades, prices)
            if not normalized or len(normalized) != len(trades) or any(d.get("action") not in ("buy", "sell") for d in normalized):
                raise ValueError("Trade failed quantity/price validation")
            for d, source in zip(normalized, trades):
                if "research_ids" in source:
                    d["research_ids"] = source["research_ids"]
                elif evidence:
                    matched = [ref for ref, syms in evidence.items() if d["symbol"] in syms]
                    if matched:
                        d["research_ids"] = matched
            info["decision_messages"] = copy.deepcopy(current)
            info["corrected_decision"] = bool(correction)
            # Corrected decision completions are executed if valid, but excluded from SFT.
            return finish("traded", trades=normalized)
        except (ValueError, TypeError, KeyError) as exc:
            call_log["error"] = str(exc)
            call_log["rejected_response"] = response
            print(f"  [Rejected {phase}] {exc}", flush=True)
            print("  [Rejected response] " + json.dumps(response, ensure_ascii=True), flush=True)
            info["retries"] += 1
            phase_attempt += 1
            if phase == "decision":
                err_msg = str(exc).rstrip(".")
                correction = (f"Correction: {err_msg}.\n" + action_context
                              + "\n" + decision_format)
            else:
                correction = corrections[phase]
            # Two retries maximum per phase; leave one generation for decisions.
            if phase_attempt >= 3 or (phase == "research" and turn >= max_turns - 2):
                return finish("failed", f"{phase}_failed")
    return finish("failed", f"{phase}_failed")


print("Decision engine loaded: bounded research, trades-only decisions, clean SFT prompts.")

Decision engine loaded: bounded research, trades-only decisions, clean SFT prompts.


### [Optional to Edit] 1.13 Training Data Utilities

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

`TradingDataCollector` records every agent episode to `sft_data.jsonl` on Google Drive.
Each line in the file is a JSON object containing:

```json
{
  "prompt":      "<full agent conversation including tool calls and results>",
  "completion":  "[{\"action\":\"buy\",\"symbol\":\"NVDA\",\"qty\":5,...}]",
  "equity":      100000.0,
  "equity_after": 101200.0,
  "pnl":         1200.0,
  "label":       "good",
  "timestamp":   1705312800
}
```

**Why store the full conversation as `prompt`?**
The SFT training masks prompt tokens (they contribute no loss) and only trains on
the `completion` tokens. Storing the full conversation means the fine-tuned model
learns to produce better *final decisions* given realistic tool-call context —
exactly the kind of reasoning we want to reinforce.

**Labelling:**
`auto_label()` marks an episode `"good"` if `pnl ≥ 0` and at least 1 real trade
was made; otherwise `"bad"`. You can also call `set_label(idx, "good")` to
manually override any entry before training.

**Workflow:**
```
run_backtest()           → fills sft_data.jsonl with unlabelled episodes
collector.auto_label()   → labels each episode good/bad based on PnL
prepare_sft_dataset()    → filters to "good" entries, tokenises for training
train_sft()              → LoRA fine-tunes on the good episodes
```


In [14]:
# =============================================================================
# Training Data Collection Utilities
# =============================================================================

class TradingDataCollector:
    """
    Logs agent episodes (conversation + decisions + outcomes) to Google Drive
    for SFT fine-tuning.

    Each episode stored in sft_data.jsonl as:
        prompt      = full agent conversation (system + tool calls + results)
        completion  = final JSON decision string
        equity      = portfolio value before this decision
        equity_after= portfolio value after trading
        pnl         = equity_after - equity
        label       = "good" | "bad" | None (set by auto_label or set_label)
    """

    def __init__(self):
        self.save_path = os.path.join(CACHE_DIR, "sft_data.jsonl")
        self.entries   = []
        self.collection_stats = {"attempted": 0, "corrected_decision": 0, "failed": 0, "no_trade": 0, "recorded": 0}
        if os.path.exists(self.save_path):
            with open(self.save_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.entries.append(json.loads(line))
            print(f"Loaded {len(self.entries)} existing training entries")

    # ── Logging ────────────────────────────────────────────────────────────

    def log(self, sft_prompt: str, sft_completion: str,
            decisions: list, equity: float,
            tool_calls: int = 0, metadata: dict = None) -> int | None:
        """
        Record one agent episode for SFT fine-tuning.

        Hard filters (drop the episode if any is true):
        - No tool was called (tool_calls <= 0)
        - No real trade (no BUY/SELL with qty > 0)
        - Any BUY/SELL decision is a validation fallback (reason == "validation_fallback")
        - Any BUY/SELL has invalid qty (missing / non-positive / non-int)
        """

        self.collection_stats["attempted"] += 1
        if metadata is not None:
            if metadata.get("corrected_decision"):
                self.collection_stats["corrected_decision"] += 1
            if metadata.get("status") == "failed":
                self.collection_stats["failed"] += 1
            if metadata.get("status") == "no_trade":
                self.collection_stats["no_trade"] += 1
            if not metadata.get("sft_eligible"):
                return None
        if not sft_prompt:
            return None

        # --- Filter 1: must have used at least one tool ---
        # We only log decisions informed by external data retrieval.
        if tool_calls <= 0:
            return None

        # --- Filter 2: must contain at least one real trade ---
        # Keep only BUY/SELL actions (HOLDs do not contribute to learning trading behavior).
        real_trades = [d for d in decisions if isinstance(d, dict) and d.get("action") in ("buy", "sell")]
        if not real_trades:
            return None

        # --- Filter 3: drop any episode that contains validation fallback trades ---
        # We do not want the model to learn from forced/invalid outputs.
        if any(d.get("reason") == "validation_fallback" for d in real_trades):
            return None

        # --- Filter 4: qty must be a positive integer for every real trade ---
        # Prevent logging malformed trades that would be rejected by the executor/validator.
        cleaned_real_trades = []
        for d in real_trades:
            qty = d.get("qty")
            if not isinstance(qty, int) or qty <= 0:
                return None
            cleaned_real_trades.append(d)

        # --- Log valid episode ---
        entry = {
            "prompt":       sft_prompt,
            "completion":   sft_completion,
            "equity":       equity,
            "equity_after": None,
            "pnl":          None,
            "label":        None,
            "timestamp":    int(time.time()),
        }

        if metadata is not None:
            entry["agent_metadata"] = metadata
        self.collection_stats["recorded"] += 1
        self.entries.append(entry)

        with open(self.save_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        return len(self.entries) - 1



    # ── Labelling ──────────────────────────────────────────────────────────

    def auto_label(self, threshold: float = 0.0, min_trades: int = 1):
        """
        Auto-label all entries that have a pnl outcome.

        Label = "good"  if pnl >= threshold AND n_actual_trades >= min_trades
        Label = "bad"   otherwise (passive episode or lost money)
        """
        labeled = 0
        for e in self.entries:
            if e.get("pnl") is not None and e["label"] is None:
                decs     = json.loads(e["completion"])
                n_trades = sum(1 for d in decs if d.get("action") in ("buy", "sell"))
                if n_trades < min_trades:
                    e["label"] = "bad"
                else:
                    e["label"] = "good" if e["pnl"] >= threshold else "bad"
                labeled += 1
        self._save_all()
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Labelled {labeled} episodes  →  {good} good, {bad} bad")

    def set_label(self, idx: int, label: str):
        """Manually override the label of a specific entry."""
        if 0 <= idx < len(self.entries):
            self.entries[idx]["label"] = label
            self._save_all()

    def get_labeled(self, label: str = "good") -> list:
        """Return all entries with the given label."""
        return [e for e in self.entries if e.get("label") == label]

    # ── Review ─────────────────────────────────────────────────────────────

    def summary(self):
        stats = self.collection_stats
        share = 100 * stats["corrected_decision"] / max(1, stats["attempted"])
        print(f"This run collection: {stats}; corrected-decision exclusion share: {share:.1f}%")
        total        = len(self.entries)
        with_outcome = sum(1 for e in self.entries if e.get("pnl") is not None)
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad  = sum(1 for e in self.entries if e.get("label") == "bad")
        print(f"Episodes: {total} | With outcome: {with_outcome} | "
              f"Good: {good} | Bad: {bad} | Unlabelled: {total-good-bad}")

    def review(self, n: int = 10):
        """Print a summary of the last n entries."""
        for i, e in enumerate(self.entries[-n:]):
            idx     = len(self.entries) - n + i
            pnl     = e.get("pnl")
            pnl_str = f"${pnl:+.2f}" if pnl is not None else "pending"
            label   = e.get("label") or "unlabelled"
            decs    = json.loads(e["completion"])
            actions = [f"{d['symbol']}:{d['action']}"
                       for d in decs if d.get("action") != "hold"][:5]
            print(f"[{idx:3d}] {label:>9} | P&L: {pnl_str:>10} | "
                  f"trades: {', '.join(actions) or 'all hold'}")
    def update_outcome(self, idx: int, equity_after: float):
        """
        Update the post-trade outcome for a logged episode.

        Args:
            idx          : index returned by log()
            equity_after : portfolio equity after trades

        Computes:
            pnl = equity_after - equity_before

        Notes:
        - If idx is None or invalid → silently ignore
        - Only updates once (prevents overwriting)
        """

        # --- Safety check ---
        if idx is None:
            return

        if not (0 <= idx < len(self.entries)):
            return

        entry = self.entries[idx]

        # --- Prevent double update ---
        if entry.get("equity_after") is not None:
            return

        entry["equity_after"] = equity_after
        entry["pnl"] = equity_after - entry["equity"]

        self._save_all()
    # ── Internal ───────────────────────────────────────────────────────────

    def _save_all(self):
        with open(self.save_path, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")


print("Training data utilities loaded.")


Training data utilities loaded.


### 1.14 Backtesting Infrastructure

> **Run this cell now, in order.** It only defines tools here; Task 4 uses them (4a collects training data, 4c compares the model before and after training).

Simulates N trading sessions using adjusted historical prices (yfinance) and
historical news (Alpaca). Decisions are made five minutes after the session close;
the resulting orders are simulated at the next session's open. Cached data is local.

#### Data flow per backtest day

```
fetch_historical_prices()           → adjusted daily OHLCV bars (yfinance, locally cached)

For each trading session:
  1. SimulatedPortfolio.execute()   → fill the prior decision at this session's open
  2. Value the portfolio at the close; mark missing held-stock prices as estimated
  3. Update the prior collector outcome if at least one order filled
  4. llm_agent_decide()             → research only data available by close + 5 minutes
  5. TradingDataCollector.log()     → record the decision; orders wait for the next open

Orders from the final session remain unfilled.
```

#### Key classes

| Class | Purpose |
|-------|---------|
| `SimulatedPortfolio` | Tracks cash, positions, equity without any real orders |
| `fetch_historical_prices()` | Downloads/caches adjusted yfinance daily bars |
| `run_backtest()` | Orchestrates the full backtest loop |

#### After the backtest

```python
collector.auto_label()   # label good/bad based on PnL
collector.summary()      # print statistics
collector.review()       # inspect recent episodes
```
Then proceed to Task 4 to train the SFT model on the "good" episodes.


In [15]:
# =============================================================================
# Backtesting Infrastructure
# =============================================================================

class SimulatedPortfolio:
    """In-memory portfolio using adjusted prices; never submits broker orders."""

    def __init__(self, initial_cash: float = 100_000.0):
        self.initial_cash = initial_cash
        self.cash = initial_cash
        self.positions = {}
        self.last_prices = {}

    def update_prices(self, prices):
        for symbol, value in prices.items():
            price = _valid_price(value)
            if price is not None:
                self.last_prices[symbol] = price

    def equity(self, prices: dict) -> float:
        self.update_prices(prices)
        return self.cash + sum(
            pos["qty"] * self.last_prices.get(symbol, pos["avg_price"])
            for symbol, pos in self.positions.items()
        )

    def get_state(self, prices: dict, timestamp: int = None) -> dict:
        estimated = sorted(symbol for symbol in self.positions if _valid_price(prices.get(symbol)) is None)
        equity = self.equity(prices)
        pos = [{"symbol": symbol, "qty": p["qty"],
                "market_value": round(p["qty"] * self.last_prices.get(symbol, p["avg_price"]), 2),
                "avg_entry_price": round(p["avg_price"], 2)}
               for symbol, p in self.positions.items()]
        return {"cash": round(self.cash, 2), "equity": round(equity, 2),
                "buying_power": round(self.cash * 2, 2), "positions": pos,
                "allowed_symbols": WHITELIST, "max_trade_value": MAX_TRADE_VALUE,
                "timestamp_unix": timestamp if timestamp is not None else int(time.time()),
                "estimated_symbols": estimated}

    def execute(self, decision: dict, prices: dict):
        action, symbol, qty = decision.get("action"), decision.get("symbol"), decision.get("qty")
        price = _valid_price(prices.get(symbol))
        if action not in ("buy", "sell") or not symbol or not isinstance(qty, int) or isinstance(qty, bool) or qty <= 0 or price is None:
            return None
        requested_qty = qty
        if action == "sell":
            pos = self.positions.get(symbol)
            if pos is None or pos["qty"] < qty:
                return None
        qty = cap_trade_quantity(qty, price)
        if qty <= 0:
            return None
        if action == "buy":
            qty = min(qty, int(self.cash / price))
            while qty > 0 and qty * price > self.cash:
                qty -= 1
            if qty <= 0:
                return None
            cost = qty * price
            self.cash -= cost
            old = self.positions.get(symbol)
            if old:
                total = old["qty"] + qty
                self.positions[symbol] = {"qty": total, "avg_price": (old["qty"] * old["avg_price"] + cost) / total}
            else:
                self.positions[symbol] = {"qty": float(qty), "avg_price": price}
            self.last_prices.setdefault(symbol, price)
        else:
            pos = self.positions.get(symbol)
            if pos is None or pos["qty"] < qty:
                return None
            self.cash += qty * price
            pos["qty"] -= qty
            if pos["qty"] <= 0:
                del self.positions[symbol]
        return {"status": "filled", "symbol": symbol, "side": action, "qty": qty,
                "requested_qty": requested_qty, "price": round(price, 2)}


def _valid_price(value):
    import math
    try:
        value = float(value)
        return value if math.isfinite(value) and value > 0 else None
    except (TypeError, ValueError):
        return None


def _backtest_calendar():
    import exchange_calendars as xcals
    return xcals.get_calendar("XNYS")


def _backtest_sessions(n_days, end_date=None, now=None):
    """Return the latest completed NYSE sessions and their UTC close times."""
    if not isinstance(n_days, int) or n_days <= 0:
        raise ValueError("n_days must be a positive integer")
    now = pd.Timestamp.now(tz="UTC") if now is None else pd.Timestamp(now)
    if now.tzinfo is None:
        raise ValueError("Backtest time must be timezone-aware")
    now = now.tz_convert("UTC")
    if end_date is not None:
        end = (pd.Timestamp(end_date).normalize().tz_localize("America/New_York") + pd.Timedelta(days=1)).tz_convert("UTC") - pd.Timedelta(microseconds=1)
        now = min(now, end)
    calendar = _backtest_calendar()
    start = (now - pd.Timedelta(days=max(60, n_days * 3 + 30))).date().isoformat()
    sessions = calendar.sessions_in_range(start, now.date().isoformat())
    completed = [(session, calendar.session_close(session)) for session in sessions
                 if calendar.session_close(session) + pd.Timedelta(minutes=5) <= now]
    return completed[-n_days:]


def fetch_historical_prices(symbols: list = None, period: str = "3mo", start=None, end=None) -> dict:
    """Fetch adjusted OHLCV daily bars; end is exclusive when supplied."""
    import hashlib
    symbols = canonicalize_symbols(WHITELIST if symbols is None else symbols)
    params = {"schema": 2, "symbols": symbols, "period": period if start is None and end is None else None,
              "start": str(start) if start is not None else None,
              "end": str(end) if end is not None else None, "auto_adjust": True}
    empty = {"dates": [], "prices": {}, "bars": {}, "timestamp": time.time(), **params}
    if not symbols:
        return empty
    digest = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]
    cache_path = os.path.join(CACHE_DIR, f"historical_bars_v2_{digest}.json")
    if os.path.exists(cache_path):
        try:
            with open(cache_path, encoding="utf-8") as f:
                cached = json.load(f)
            if all(cached.get(k) == v for k, v in params.items()) and time.time() - cached.get("timestamp", 0) < 86400:
                return cached
        except (OSError, ValueError, TypeError):
            pass
    kwargs = {"auto_adjust": True, "progress": False, "group_by": "column"}
    if start is None and end is None:
        kwargs["period"] = period
    else:
        kwargs.update(start=start, end=end)
    try:
        frame = yf.download(symbols, **kwargs)
        if frame is None or frame.empty:
            return empty
        if not isinstance(frame.columns, pd.MultiIndex) and len(symbols) != 1:
            print("  [Prices] Cannot identify symbols in historical data.")
            return empty
        bars, prices = {}, {}
        for symbol in symbols:
            rows = {}
            for index, row in frame.iterrows():
                record = {}
                for field in ("Open", "High", "Low", "Close", "Volume"):
                    try:
                        val = row[(field, symbol)] if isinstance(frame.columns, pd.MultiIndex) else row[field]
                        val = float(val)
                        record[field.lower()] = val if pd.notna(val) and abs(val) != float("inf") and val >= 0 else None
                    except (KeyError, TypeError, ValueError):
                        record[field.lower()] = None
                if any(value is not None for value in record.values()):
                    rows[pd.Timestamp(index).strftime("%Y-%m-%d")] = record
            if rows:
                bars[symbol] = rows
                prices[symbol] = {date: row["close"] for date, row in rows.items() if _valid_price(row["close"]) is not None}
        result = {"dates": sorted({d for rows in bars.values() for d in rows}),
                  "prices": prices, "bars": bars, "timestamp": time.time(), **params}
        if result["dates"]:
            try:
                with open(cache_path, "w", encoding="utf-8") as f:
                    json.dump(result, f)
            except OSError as exc:
                print(f"  [Prices] Could not save historical cache: {exc}")
        return result
    except Exception as exc:
        print(f"  [Prices] Historical download failed: {exc}")
        return empty


def _completed_session_for(as_of):
    cutoff = pd.Timestamp(as_of)
    if cutoff.tzinfo is None:
        raise ValueError("Historical cutoff must include a timezone")
    cutoff = cutoff.tz_convert("UTC")
    calendar = _backtest_calendar()
    local_date = cutoff.tz_convert("America/New_York").date().isoformat()
    session = calendar.date_to_session(local_date, direction="previous")
    if calendar.session_close(session) > cutoff:
        session = calendar.previous_session(session)
    return session.strftime("%Y-%m-%d")


def historical_market_tool(name, arguments, as_of, price_data):
    """Use only bars completed by the trusted decision timestamp."""
    if not isinstance(price_data, dict) or "bars" not in price_data:
        return "Historical market data unavailable."
    symbol = arguments.get("symbol") or arguments.get("ticker") or arguments.get("symbols")
    if isinstance(symbol, list):
        symbol = symbol[0] if symbol else None
    if not isinstance(symbol, str) or not symbol.strip():
        return "Historical market data unavailable: no symbol."
    symbol = symbol.strip().upper()
    day = _completed_session_for(as_of)
    all_rows = price_data["bars"].get(symbol, {})
    if _valid_price(all_rows.get(day, {}).get("close")) is None:
        return f"Historical market data unavailable for {symbol} on {day}."
    rows = [(d, v) for d, v in sorted(all_rows.items()) if d <= day and _valid_price(v.get("close")) is not None]
    if name == "get_price_history":
        period = str(arguments.get("period", "5d"))
        if period == "5d":
            rows = rows[-5:]
        elif period in ("1mo", "3mo", "6mo", "1y"):
            months = {"1mo": 1, "3mo": 3, "6mo": 6, "1y": 12}[period]
            start = (pd.Timestamp(day) - pd.DateOffset(months=months)).strftime("%Y-%m-%d")
            rows = [(d, v) for d, v in rows if d >= start]
        else:
            return f"Historical price period unavailable: {period}"
        lines = [f"Price history for {symbol} ({period}):"]
        for date, v in rows:
            if any(_valid_price(v.get(k)) is None for k in ("open", "high", "low", "close")) or v.get("volume") is None:
                continue
            lines.append(f"  {date}: O={v['open']:.2f} H={v['high']:.2f} L={v['low']:.2f} C={v['close']:.2f} V={int(v['volume'])}")
        return "\n".join(lines) if len(lines) > 1 else f"Historical price history unavailable for {symbol}."
    if name in ("get_rsi", "get_macd"):
        months = 3 if name == "get_rsi" else 6
        start = (pd.Timestamp(day) - pd.DateOffset(months=months)).strftime("%Y-%m-%d")
        close = pd.Series([v["close"] for d, v in rows if d >= start], dtype=float)
        if name == "get_rsi":
            try:
                period = int(arguments.get("time_period", 14))
            except (TypeError, ValueError):
                return "Historical RSI unavailable: invalid period."
            if period < 1 or len(close) < period + 1:
                return f"Historical RSI unavailable for {symbol}."
            delta = close.diff()
            gain = delta.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
            loss = (-delta.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
            value = float((100 - 100 / (1 + gain / loss)).iloc[-1])
            if pd.isna(value):
                return f"Historical RSI unavailable for {symbol}."
            signal = "overbought" if value > 70 else "oversold" if value < 30 else "neutral"
            return f"RSI({period}) for {symbol}: {value:.1f} ({signal}) [{day}]"
        if len(close) < 26:
            return f"Historical MACD unavailable for {symbol}."
        macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
        signal = macd.ewm(span=9, adjust=False).mean()
        m, s = float(macd.iloc[-1]), float(signal.iloc[-1])
        return f"MACD for {symbol} [{day}]: MACD={m:.4f}, Signal={s:.4f}, Hist={m-s:.4f} ({'bullish' if m > s else 'bearish'})"
    if name == "get_global_quote":
        current = rows[-1][1]
        price = current["close"]
        previous_day = _backtest_calendar().previous_session(pd.Timestamp(day)).strftime("%Y-%m-%d")
        previous = _valid_price(all_rows.get(previous_day, {}).get("close"))
        change = f"{price-previous:+.2f} ({(price/previous-1)*100:+.2f}%)" if previous else "unavailable"
        volume = f"{int(current['volume']):,}" if current.get("volume") is not None else "unavailable"
        return f"Quote {symbol}: ${price:.2f} | change: {change} | vol: {volume} [{day} close]"
    return f"Historical version of tool '{name}' is unavailable."


def run_backtest(tok, model, collector=None, n_days: int = 20,
                 initial_cash: float = 100_000.0, verbose: bool = True, end_date=None) -> dict:
    """Decide five minutes after each close and fill at the next session open."""
    import time as _backtest_time
    setup_started = _backtest_time.perf_counter()
    if verbose:
        print("[Backtest setup] Preparing market sessions...", flush=True)
    sessions = _backtest_sessions(n_days, end_date=end_date)
    if not sessions:
        raise ValueError("No completed market sessions are available for this backtest")
    first = pd.Timestamp(sessions[0][0]).date()
    last = pd.Timestamp(sessions[-1][0]).date()
    start = (first - datetime.timedelta(days=240)).isoformat()
    end = (last + datetime.timedelta(days=1)).isoformat()
    if verbose:
        print("[Backtest setup] Loading historical bars...", flush=True)
    data = fetch_historical_prices(start=start, end=end)
    if not data.get("bars"):
        raise ValueError("Historical price data is unavailable for the backtest")
    portfolio = SimulatedPortfolio(initial_cash)
    equity_history, all_trades, skipped_orders = [], [], []
    pending_decisions, pending_idx = [], None
    session_diagnostics = []
    if verbose:
        print(f"[Backtest setup] Ready in {_backtest_time.perf_counter() - setup_started:.1f}s")
        print(f"BACKTEST {first} → {last} ({len(sessions)} sessions); decisions after close, fills next open")
    for session_number, (session, close_time) in enumerate(sessions, 1):
        session_started = _backtest_time.perf_counter()
        day = pd.Timestamp(session).strftime("%Y-%m-%d")
        if verbose:
            print(f"[Session {session_number}/{len(sessions)}] {day}: starting fills, valuation and research...", flush=True)
        day_bars = {sym: data["bars"].get(sym, {}).get(day, {}) for sym in WHITELIST}
        opens = {sym: row.get("open") for sym, row in day_bars.items() if _valid_price(row.get("open")) is not None}
        closes = {sym: row.get("close") for sym, row in day_bars.items() if _valid_price(row.get("close")) is not None}
        executed = 0
        for decision in pending_decisions:
            result = portfolio.execute(decision, opens)
            if result:
                all_trades.append({**result, "date": day})
                executed += 1
                if result["qty"] < result["requested_qty"]:
                    print(f"  [Reduced] {result['symbol']}: {result['requested_qty']} to {result['qty']} shares (trade-value limit or available cash)")
            elif decision.get("action") in ("buy", "sell"):
                symbol = decision.get("symbol")
                reason = "missing opening price" if symbol not in opens else "trade-value limit, insufficient cash, or insufficient holdings"
                skipped_orders.append({"date": day, "decision": decision, "reason": reason})
                print(f"  [Skipped] {symbol}: {reason}")
        cutoff = close_time + pd.Timedelta(minutes=5)
        state = portfolio.get_state(closes, timestamp=int(cutoff.timestamp()))
        estimated = state["estimated_symbols"]
        equity_history.append({"date": day, "equity": state["equity"],
                               "estimated": bool(estimated), "estimated_symbols": estimated})
        if collector is not None and pending_idx is not None and executed:
            collector.update_outcome(pending_idx, state["equity"])
        outcome = llm_agent_decide(
            tok, model, state, universe=WHITELIST, date=cutoff.isoformat(), historical_data=data)
        decisions, prompt, completion, tool_calls = outcome
        meta = getattr(outcome, "diagnostics", None)
        session_diagnostics.append({"date": day, "equity": state["equity"],
                                    "positions": state["positions"], **(meta or {"status": "unknown"})})
        if collector is not None:
            options = {"metadata": meta} if meta is not None else {}
            pending_idx = collector.log(prompt, completion, decisions, state["equity"], tool_calls, **options)
        else:
            pending_idx = None
        pending_decisions = decisions
        if verbose:
            suffix = f" | estimated: {', '.join(estimated)}" if estimated else ""
            session_elapsed = _backtest_time.perf_counter() - session_started
            print(f"{day}: Equity ${state['equity']:,.2f} | Fills: {executed}{suffix} | Session: {session_elapsed:.1f}s", flush=True)
    final_equity = equity_history[-1]["equity"]
    total_return = (final_equity - initial_cash) / initial_cash * 100
    any_estimated = any(entry["estimated"] for entry in equity_history)
    failed = sum(d["status"] == "failed" for d in session_diagnostics)
    no_trade = sum(d["status"] == "no_trade" for d in session_diagnostics)
    cited_sessions = sum(bool(d.get("evidence")) and d["status"] == "traded" for d in session_diagnostics)
    trade_count = sum(d.get("trade_decisions", 0) for d in session_diagnostics)
    verified_count = sum(d.get("verified_reference_decisions", 0) for d in session_diagnostics)
    quality = {"trade_decisions": trade_count, "verified_reference_decisions": verified_count,
               "verified_reference_rate_pct": 100 * verified_count / trade_count if trade_count else None,
               "sessions": len(sessions), "failed_sessions": failed,
               "failure_rate_pct": 100 * failed / len(sessions),
               "no_trade_sessions": no_trade, "no_trade_rate_pct": 100 * no_trade / len(sessions),
               "trade_sessions_with_verified_references": cited_sessions,
               "cap_hits": sum(c.get("cap_hit", False) for d in session_diagnostics for c in d.get("calls", [])),
               "corrected_decision_sessions": sum(d.get("corrected_decision", False) for d in session_diagnostics)}
    if verbose:
        print(f"AGENT QUALITY | Failures: {failed}/{len(sessions)} ({quality['failure_rate_pct']:.1f}%) | "
              f"Researched no-trade: {no_trade}/{len(sessions)} ({quality['no_trade_rate_pct']:.1f}%) | "
              f"Cap hits: {quality['cap_hits']}")
        print(f"RESULTS | Final: ${final_equity:,.2f} | Return: {total_return:+.2f}% | Trades: {len(all_trades)}" + (" | Includes estimated valuations" if any_estimated else ""))
        print("Final-session decisions remain unfilled.")
    return {"equity_history": equity_history, "trades": all_trades, "total_return": total_return,
            "final_equity": final_equity, "estimated": any_estimated,
            "skipped_orders": skipped_orders,
            "session_diagnostics": session_diagnostics, "agent_quality": quality,
            "unfilled_decisions": [d for d in pending_decisions if d.get("action") in ("buy", "sell")]}


print("Backtesting infrastructure loaded.")


Backtesting infrastructure loaded.


### 1.15 Test Agent Decision Engine

Runs a quick smoke test of `llm_agent_decide()` using only the tools that are
currently implemented (i.e. registered with a non-None `func`).

**At this point (before Task 2):**
- `get_news` is available — the agent can fetch financial news
- `get_price_history`, `get_rsi`, `get_macd`, `get_global_quote` are not yet available

After completing Task 2 and registering your tools, re-run this cell to see
the agent use all of your implementations.


In [26]:
# Quick smoke test — only tools implemented so far will be available
print("Available tools before Task 2:")
list_tools()

print("\nRunning agent on 5-symbol subset...")
state = get_state(tc)
decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
    tok, model, state, universe=WHITELIST[:5]
)

print(f"\nDecisions ({len(decisions)}):")
for d in decisions:
    sym    = d.get("symbol", "?")
    act    = d["action"].upper()
    qty    = d.get("qty", "-")
    reason = d.get("reason", "")[:60]
    print(f"  [{sym}] {act:5s}  qty={str(qty):>4}  {reason}")

print(f"\nSFT prompt  : {len(sft_prompt):,} chars")
print(f"SFT completion: {len(sft_completion):,} chars")


Available tools before Task 2:
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news for specific stocks. Pass all requested symbols in one call. Matching requests are cached; backtests restrict news to their decision time.
  get_price_history    OK             Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              OK             Get RSI indicator (>70 overbought, <30 oversold) via yfinance
  get_macd             OK             Get MACD trend indicator via yfinance
  get_global_quote     OK             Get real-time price quote via Alpaca Market Data

Running agent on 5-symbol subset...
  [Generation 1/6] Starting research, attempt 1...
  [Generation] research: 2.4s | input=578 | output=22/180 | cap_hit=False | components={'instructions': 420, 'account_state': 104, 'tool_results': 0, 'assistant_calls': 0, 'phase': 30, 'retry': 0, 'extra_cont

### Task 1 Complete!

You now have all core infrastructure running:
- LLM model loaded
- Alpaca trading client connected
- News cached to Google Drive
- Decision engine ready
- Training data collector ready (for Task 4)
- Backtesting engine ready (for Task 4)

**Next: Implement Tasks 2-5 below.**

---

## Task 2: [TODO] Implement MCP Tool Functions

### Objective

Implement the **data retrieval tools** that the trading agent uses to research stocks
before making decisions. When the agent outputs a tool call, `dispatch_tool()` routes
it to your function — so the quality of your implementations directly determines
how well-informed the agent's decisions are.

### Tool Registry Reference

All tool functions you implement in this task must be registered using the `register_tool()`
function from **Section 1.10**. After you write each function, call:

```python
register_tool(
    name        = "your_tool_name",
    description = "brief description for the agent",
    params      = {"param": "type — description"},
    func        = your_function,
)
```

See Section 1.10 for the complete Tool Registry API documentation.

### How a tool call reaches your code

```
Agent outputs:
  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA", "time_period": 14}}</tool_call>

dispatch_tool("get_rsi", {"symbol": "NVDA", "time_period": 14})
  └─ _tool_registry["get_rsi"]["func"]("NVDA", time_period=14)
       └─ YOUR get_rsi("NVDA", time_period=14)   ← implement this
            └─ returns: "RSI(14) for NVDA: 62.3 (neutral) [2024-01-15]"
```

### [TODO] Required: implement and register these tools

**Implementation order:** Start with `get_news_for_symbols` (Section 1.9) as it demonstrates the complete pattern: API call → error handling → caching → formatting. Then implement the remaining functions following the same pattern.

| Function | Data source | What to implement |
|----------|------------|-------------------|
| `get_news_for_symbols(symbols, limit, date)` | **Alpaca** | News with caching |
| `get_price_history(symbol, period)` | **yfinance** | Provided: recent OHLCV table |
| `get_rsi(symbol, time_period)` | **yfinance** | RSI value + signal |
| `get_macd(symbol)` | **yfinance** | MACD, signal, histogram (computed from daily closes) |
| `get_global_quote(symbol)` | **Alpaca** | Live price, change, volume |

> Alpha Vantage made `NEWS_SENTIMENT` and `MACD` premium-only, so news comes from Alpaca (Section 1.9)
> and MACD is computed from yfinance prices. RSI and GLOBAL_QUOTE still work with the free Alpha Vantage key.

After implementing each function you **must** call `register_tool()` to update the
registry — otherwise the agent won't know the tool is ready.

### Alpha Vantage quick reference

```python
# RSI
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "RSI", "symbol": symbol,
    "interval": "daily", "time_period": time_period,
    "series_type": "close", "apikey": ALPHA_VANTAGE_API_KEY,
})
data = resp.json()
# data["Technical Analysis: RSI"] -> {date: {"RSI": "65.32"}, ...}

# Global Quote
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": symbol,
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Global Quote"] -> {"05. price": "182.63", "09. change": "+1.20", ...}
```

### yfinance quick reference

```python
import yfinance as yf
ticker = yf.Ticker("AAPL")
hist   = ticker.history(period="5d")   # pandas DataFrame
# columns: Open, High, Low, Close, Volume
# index:   DatetimeIndex

# MACD from daily closes (Alpha Vantage's MACD is premium-only)
close  = ticker.history(period="6mo")["Close"]
macd   = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
signal = macd.ewm(span=9, adjust=False).mean()
hist   = macd - signal
```

### Optional: add your own custom tool

Implement any additional data source (earnings, options flow, sector ETF performance,
macroeconomic indicators, etc.) and register it so the agent can call it automatically:

```python
def get_earnings(symbol: str) -> str:
    """Fetch latest quarterly earnings from Alpha Vantage."""
    # function="EARNINGS" -> data["quarterlyEarnings"][0]
    ...

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# Agent now calls get_earnings automatically when it wants earnings data
```

### Grading criteria

- [ ] `get_news_for_symbols` implements full caching + API call pattern. *Evidence: Test 1.9 (the repeated query prints `Cache hit`).*
- [ ] At least **2** Alpha Vantage functions implemented; each handles API errors gracefully (try/except); all registered with `register_tool(..., func=your_fn)`. *Evidence: Test Task 2 output and `list_tools()` in 1.15.*
- `get_price_history` is provided. Optional custom tools are welcome; describe them in your report.


In [25]:
# =============================================================================
# TODO: Task 2 — Implement MCP Tool Functions
# Each function is called by dispatch_tool() when the agent requests it.
# After implementing a function, update its registry entry so the agent can use it.
# =============================================================================


# ── Required: get_price_history ───────────────────────────────────────────────

def get_price_history(symbol: str, period: str = "5d") -> str:
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period)

        if hist.empty:
            return f"No price history available for {symbol}."

        lines = [f"Price history for {symbol} ({period}):"]

        for date, row in hist.iterrows():
            d = date.strftime("%Y-%m-%d")
            o = row["Open"]
            h = row["High"]
            l = row["Low"]
            c = row["Close"]
            v = int(row["Volume"])

            lines.append(
                f"  {d}: O={o:.2f} H={h:.2f} L={l:.2f} C={c:.2f} V={v}"
            )

        return "\n".join(lines)

    except Exception as e:
        return f"Price history fetch failed for {symbol}: {e}"


register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = get_price_history,   # update func here after implementing above
)


# ── Required: get_rsi ─────────────────────────────────────────────────────────

def get_rsi(symbol: str = None, interval: str = "daily", time_period: int = 14, **kwargs) -> str:
    """
    Compute RSI (Relative Strength Index) for a symbol from yfinance daily closes.

    RSI > 70  →  overbought (potential sell signal)
    RSI < 30  →  oversold   (potential buy signal)
    30–70     →  neutral

    Returns:
        e.g. "RSI(14) for NVDA: 68.5 (neutral) [2024-01-15]"
    """
    try:
        if symbol is None:
            symbol = kwargs.get("symbols") or kwargs.get("ticker")
        if isinstance(symbol, list):
            symbol = symbol[0] if symbol else None
        if not symbol or not isinstance(symbol, str):
            return "Error: No valid symbol provided to get_rsi."

        symbol = symbol.strip().upper()
        try:
            time_period = int(time_period)
        except (ValueError, TypeError):
            time_period = 14

        ticker = yf.Ticker(symbol)
        hist_df = ticker.history(period="3mo")
        if hist_df.empty or len(hist_df) < time_period + 1:
            return f"No price history available to compute RSI for {symbol}."

        close = hist_df["Close"]
        delta = close.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)

        # Standard Wilder's smoothing
        avg_gain = gain.ewm(com=time_period - 1, min_periods=time_period).mean()
        avg_loss = loss.ewm(com=time_period - 1, min_periods=time_period).mean()

        rs = avg_gain / avg_loss
        rsi = 100.0 - (100.0 / (1.0 + rs))

        latest_date = hist_df.index[-1].strftime("%Y-%m-%d")
        val = float(rsi.iloc[-1])

        if val > 70:
            signal = "overbought"
        elif val < 30:
            signal = "oversold"
        else:
            signal = "neutral"

        return f"RSI({time_period}) for {symbol}: {val:.1f} ({signal}) [{latest_date}]"

    except Exception as e:
        return f"RSI computation failed for {symbol}: {e}"


register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via yfinance",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = get_rsi,
)


# ── Required: get_macd ────────────────────────────────────────────────────────

def get_macd(symbol: str, interval: str = "daily") -> str:
    """
    Fetch MACD (Moving Average Convergence Divergence) for a symbol.

    MACD > Signal  →  bullish crossover
    MACD < Signal  →  bearish crossover

    Alpha Vantage's MACD is premium-only, so compute it from yfinance daily closes
    (see "yfinance quick reference" in the Task 2 description):
        MACD   = EMA12(close) - EMA26(close)
        Signal = EMA9(MACD)
        Hist   = MACD - Signal

    Returns:
        e.g. "MACD for NVDA [2024-01-15]: MACD=0.1234, Signal=0.0987, Hist=0.0247 (bullish)"
    """
    try:
        ticker = yf.Ticker(symbol)
        hist_df = ticker.history(period="6mo")
        if hist_df.empty:
            return f"No price history available to compute MACD for {symbol}."

        close = hist_df["Close"]
        macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
        signal = macd.ewm(span=9, adjust=False).mean()
        hist = macd - signal

        latest_date = hist_df.index[-1].strftime("%Y-%m-%d")
        m_val = float(macd.iloc[-1])
        s_val = float(signal.iloc[-1])
        h_val = float(hist.iloc[-1])

        trend = "bullish" if m_val > s_val else "bearish"

        return f"MACD for {symbol} [{latest_date}]: MACD={m_val:.4f}, Signal={s_val:.4f}, Hist={h_val:.4f} ({trend})"

    except Exception as e:
        return f"MACD computation failed for {symbol}: {e}"


register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via yfinance",
    params      = {"symbol": "str"},
    func        = get_macd,
)


# ── Required: get_global_quote ────────────────────────────────────────────────

def get_global_quote(symbol: str = None, **kwargs) -> str:
    """
    Fetch real-time price quote for a symbol from Alpaca Market Data API.

    Alpaca endpoint:
        https://data.alpaca.markets/v2/stocks/{symbol}/snapshot

    Response keys:
        latestTrade: {"p": price, ...}
        dailyBar: {"c": close, "v": volume, ...}
        prevDailyBar: {"c": prev_close, ...}

    Returns:
        e.g. "Quote AAPL: $186.10 | change: +1.20 (+0.65%) | vol: 52,341,200"
    """
    try:
        if symbol is None:
            symbol = kwargs.get("symbols") or kwargs.get("ticker")
        if isinstance(symbol, list):
            symbol = symbol[0] if symbol else None
        if not symbol or not isinstance(symbol, str):
            return "Error: No valid symbol provided to get_global_quote."

        symbol = symbol.strip().upper()

        url = f"https://data.alpaca.markets/v2/stocks/{symbol}/snapshot"
        headers = {
            "APCA-API-KEY-ID": ALPACA_API_KEY,
            "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY,
        }
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            return f"Could not fetch quote for {symbol}: HTTP {resp.status_code} ({resp.text[:100]})"

        data = resp.json()
        latest_trade = data.get("latestTrade") or {}
        daily_bar = data.get("dailyBar") or {}
        prev_bar = data.get("prevDailyBar") or {}

        price = float(latest_trade.get("p") or daily_bar.get("c") or 0.0)
        volume = int(daily_bar.get("v") or 0)
        prev_close = float(prev_bar.get("c") or 0.0)

        if prev_close > 0 and price > 0:
            change = price - prev_close
            change_pct = (change / prev_close) * 100
        else:
            change = 0.0
            change_pct = 0.0

        change_sign = f"+{change:.2f}" if change > 0 else f"{change:.2f}"
        return f"Quote {symbol}: ${price:.2f} | change: {change_sign} ({change_pct:+.2f}%) | vol: {volume:,}"

    except Exception as e:
        return f"Global quote fetch failed for {symbol}: {e}"


register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpaca Market Data",
    params      = {"symbol": "str"},
    func        = get_global_quote,
)


# =============================================================================
# =============================================================================
# Optional C: search_market_news -- agent-driven stock discovery
# =============================================================================
# Instead of researching pre-defined tickers, this tool fetches the latest
# market-wide news from Alpaca WITHOUT specifying any symbols. The agent reads the
# headlines and discovers interesting companies on its own.
#
# Alpaca News API (no symbols= argument; same URL and headers as Section 1.9):
#   params: limit=20 (max 50); no topic filter and no sentiment scores
#   response: data["news"] -> list of articles, each with "symbols":
#     ["NVDA", "AMD"]
#
# def search_market_news(topic: str = "technology") -> str:
#     """
#     Fetch the latest market-wide news; return top mentioned tickers + headlines.
#
#     Args:
#         topic : optional keyword (e.g. "technology", "earnings", "ipo");
#                 Alpaca has no topic filter, so filter the headlines yourself
#     Returns:
#         Formatted string, e.g.:
#         "Top mentioned tickers for topic=technology:\n"
#         "  NVDA: 5 articles\n"
#         "  MSFT: 3 articles\n"
#         "Headlines:\n  - Nvidia announces next-gen GPU ...\n"
#     """
#     url = "https://data.alpaca.markets/v1beta1/news"
#     params = {"limit": 20}   # headers: your Alpaca keys, as in Section 1.9
#     # TODO: call API, filter by topic, count symbols across articles
#     raise NotImplementedError("TODO: implement search_market_news()")
#
# register_tool(
#     name        = "search_market_news",
#     description = (
#         "Fetch the latest market-wide news (no ticker needed) and the most "
#         "mentioned tickers. Optional topic keyword, e.g. technology, earnings, ipo."
#     ),
#     params      = {"topic": "str"},
#     func        = search_market_news,
# )

# Optional: add your own custom tool
# =============================================================================
#
# def get_MY_tool(symbol: str) -> str:
#     """Your description here."""
#     # ... implementation ...
#     return formatted_string
#
# register_tool(
#     name        = "get_MY_tool",
#     description = "One-line description for the agent",
#     params      = {"symbol": "str — stock ticker"},
#     func        = get_MY_tool,
# )


# =============================================================================
# get_extra_context
# =============================================================================
# This function aggregates data from your MCP tools into a single context string
# that gets passed to llm_agent_decide() in the trading loop (Task 5, Step 3).
#
# It calls get_price_history, get_rsi, get_macd, and get_global_quote for each
# symbol, collects the results, and joins them into one big string.
# If you implemented Optional C (search_market_news), call it here too.
# =============================================================================

def get_extra_context(symbols: list) -> str:
    """
    Aggregate outputs from all implemented MCP tools for the given symbols.

    For each symbol in the list, call:
      - get_price_history(symbol)
      - get_rsi(symbol)
      - get_macd(symbol)
      - get_global_quote(symbol)

    Collect results into a single formatted string. Each tool call should be
    wrapped in try/except so one failure doesn't block the rest.

    If you implemented search_market_news (Optional C), also call it with a
    relevant topic (e.g. "technology") and append the result.

    Args:
        symbols : list of ticker strings, e.g. ["AAPL", "NVDA"]

    Returns:
        Combined string of all tool outputs, separated by newlines.
        Returns "" if all calls fail.

    Example output:
        === Extra Context for AAPL ===
        Price history for AAPL (5d):
          2024-01-15: O=185.00 H=186.50 L=184.20 C=186.10 V=52341200
          ...
        RSI(14) for AAPL: 68.5 (neutral)
        MACD for AAPL: MACD=0.12, Signal=0.10, Hist=0.02 (bullish)
        Quote AAPL: 86.10 | change: +1.20 (+0.65%) | vol: 52,341,200
        ============================================

    Hints:
        tools = [get_price_history, get_rsi, get_macd, get_global_quote]
        for symbol in symbols:
            for tool in tools:
                try: results.append(tool(symbol))
                except: pass
    """
    tools = [get_price_history, get_rsi, get_macd, get_global_quote]
    blocks = []

    for symbol in symbols:
        lines = [f"=== Extra Context for {symbol} ==="]
        for tool in tools:
            try:
                res = tool(symbol)
                if res:
                    lines.append(res)
            except Exception:
                pass
        lines.append("=" * 44)
        if len(lines) > 2:
            blocks.append("\n".join(lines))

    return "\n\n".join(blocks)

### Test Task 2

In [18]:
# Test your implementations
print("Testing MCP Tool Functions...")
print("=" * 60)

for func_name, func in [("get_rsi", get_rsi), ("get_macd", get_macd),
                          ("get_price_history", get_price_history),
                          ("get_global_quote", get_global_quote)]:
    try:
        result = func("AAPL")
        print(f"{func_name}('AAPL'):")
        print(f"  {result[:200]}")
    except NotImplementedError:
        print(f"{func_name}: Not yet implemented")
    except Exception as e:
        print(f"{func_name}: Error - {e}")

print("" + "=" * 60)

# Test get_extra_context (aggregates all tool outputs)
try:
    extra = get_extra_context(["AAPL"])
    assert isinstance(extra, str), f"Expected str, got {type(extra)}"
    print(f"get_extra_context(['AAPL']):")
    print(f"  Length: {len(extra)} chars")
    print(f"  Preview: {extra[:300]}")
except NotImplementedError:
    print("get_extra_context: Not yet implemented")
except Exception as e:
    print(f"get_extra_context: Error - {e}")

# Test that get_extra_context handles multiple symbols
try:
    extra_multi = get_extra_context(["AAPL", "NVDA"])
    assert isinstance(extra_multi, str), f"Expected str, got {type(extra_multi)}"
    assert len(extra_multi) >= len(extra), "Multi-symbol result should be >= single-symbol"
    print(f"get_extra_context(['AAPL', 'NVDA']):")
    print(f"  Length: {len(extra_multi)} chars")
except NotImplementedError:
    pass
except Exception as e:
    print(f"get_extra_context multi-symbol: Error - {e}")

print("" + "=" * 60)
print("Task 2 testing complete.")


Testing MCP Tool Functions...
get_rsi('AAPL'):
  RSI(14) for AAPL: 51.5 (neutral) [2026-11-06]
get_macd('AAPL'):
  MACD for AAPL [2026-11-06]: MACD=2.0677, Signal=2.8364, Hist=-0.7687 (bearish)
get_price_history('AAPL'):
  Price history for AAPL (5d):
  2026-11-02: O=342.67 H=343.80 L=335.82 C=338.32 V=64757700
  2026-11-03: O=325.58 H=334.50 L=325.33 C=331.90 V=63160000
  2026-11-04: O=341.57 H=354.50 L=340.99 C=349.51
get_global_quote('AAPL'):
  Quote AAPL: $342.41 | change: -2.19 (-0.64%) | vol: 1,313,258
get_extra_context(['AAPL']):
  Length: 596 chars
  Preview: === Extra Context for AAPL ===
Price history for AAPL (5d):
  2026-11-02: O=342.67 H=343.80 L=335.82 C=338.32 V=64757700
  2026-11-03: O=325.58 H=334.50 L=325.33 C=331.90 V=63160000
  2026-11-04: O=341.57 H=354.50 L=340.99 C=349.51 V=89637700
  2026-11-05: O=353.89 H=355.38 L=342.06 C=344.17 V=75678
get_extra_context(['AAPL', 'NVDA']):
  Length: 1200 chars
Task 2 testing complete.


---
## Task 3: [TODO] RAG Memory System

### Objective
Implement a memory system that allows the trading agent to **learn from past trades**.

Two components:
1. **`TradingMemory`** class - stores situation-reflection pairs, retrieves semantically similar experiences using sentence embeddings and FAISS
2. **`reflect_on_trade()`** function - uses LLM to generate a lesson learned after a trade

### How it works

```
Trade executed -> Wait N loops -> Compare equity -> LLM reflection -> Store in memory
                                                                           |
Next decision <- Retrieve similar past reflections via semantic search <--------------+
```

### Vector Memory Retrieval
Saved situations are embedded with a local sentence-transformer and indexed with FAISS.
Cosine similarity ranks memories; the configured threshold filters weak matches.
The original JSON file remains the source of truth and the index is rebuilt when needed.
This implements the vector database RAG challenge in place of BM25.

### Grading Criteria
- [ ] TradingMemory correctly stores and loads memories; semantic retrieval returns relevant past reflections (vector RAG challenge replaces BM25); edge cases handled (empty memory, no matching results). *Evidence: Test Task 3, and the first Task 5 iterations (memory is still empty there).*
- [ ] reflect_on_trade generates meaningful short reflections: each names the stock and the action and says whether the trade gained or lost. *Evidence: Test Task 3.*


### 3a. TradingMemory Class

In [19]:
class TradingMemory:
    """JSON-backed reflections with a rebuildable semantic FAISS index."""

    MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

    def __init__(self, memory_file: str = "trading_memory.json", min_similarity=None):
        self.memory_file = memory_file if memory_file is not None else os.path.join(CACHE_DIR, "trading_memory.json")
        self.min_similarity = float(globals().get("MEMORY_MIN_SIMILARITY", 0.35) if min_similarity is None else min_similarity)
        if not -1 <= self.min_similarity <= 1:
            raise ValueError("Memory similarity threshold must be between -1 and 1")
        self.memories = []
        self._snapshot = None
        self._model = None
        self._index = None
        self._refresh()
        if self.memories:
            self._rebuild_index()

    def _read_file(self):
        import hashlib
        try:
            with open(self.memory_file, "rb") as f:
                raw = f.read()
        except FileNotFoundError:
            return None, []
        try:
            entries = json.loads(raw.decode("utf-8"))
            if not isinstance(entries, list) or any(
                not isinstance(m, dict) or not isinstance(m.get("situation"), str)
                or not isinstance(m.get("reflection"), str) for m in entries
            ):
                raise ValueError("expected a list of situation/reflection objects")
        except (UnicodeDecodeError, ValueError, TypeError) as exc:
            raise RuntimeError(f"Cannot read memory file {self.memory_file}; existing data was not changed: {exc}") from exc
        return hashlib.sha256(raw).hexdigest(), entries

    def _refresh(self):
        snapshot, entries = self._read_file()
        if snapshot != self._snapshot:
            self.memories = entries
            self._snapshot = snapshot
            self._index = None

    def _ensure_backend(self):
        if self._model is None:
            try:
                import faiss
                from sentence_transformers import SentenceTransformer
                self._model = SentenceTransformer(self.MODEL_NAME, device="cpu")
            except Exception as exc:
                raise RuntimeError(
                    "Semantic memory unavailable: install compatible sentence-transformers and "
                    "faiss-cpu, and ensure the embedding model is downloaded or accessible. "
                    f"Existing memories were preserved. ({exc})"
                ) from exc
        return self._model

    def _vectors(self, texts):
        import numpy as np
        model = self._ensure_backend()
        try:
            values = np.asarray(model.encode(texts, normalize_embeddings=True,
                                              convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)
            if values.ndim != 2 or values.shape[0] != len(texts) or not np.isfinite(values).all():
                raise ValueError("invalid embedding matrix")
            norms = np.linalg.norm(values, axis=1, keepdims=True)
            if (norms <= 0).any():
                raise ValueError("embedding has zero norm")
            return np.ascontiguousarray(values / norms, dtype=np.float32)
        except Exception as exc:
            raise RuntimeError(f"Semantic memory embedding failed; existing memories were preserved: {exc}") from exc

    def _rebuild_index(self):
        self._ensure_backend()
        import faiss
        vectors = self._vectors([m["situation"] for m in self.memories])
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        self._index = index

    def _save(self):
        """Atomically persist the current list and invalidate the derived index."""
        import hashlib
        import tempfile
        raw = json.dumps(self.memories, ensure_ascii=False, indent=2).encode("utf-8")
        folder = os.path.dirname(os.path.abspath(self.memory_file))
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(mode="wb", dir=folder, prefix=".memory-", suffix=".tmp", delete=False) as f:
                temp_path = f.name
                f.write(raw)
            os.replace(temp_path, self.memory_file)
            temp_path = None
        finally:
            if temp_path is not None and os.path.exists(temp_path):
                os.unlink(temp_path)
        self._snapshot = hashlib.sha256(raw).hexdigest()
        self._index = None

    def add_memory(self, situation: str, reflection: str, trade_context: dict = None):
        """Persist a reflection, keeping existing JSON fields and optional metadata."""
        if not isinstance(situation, str) or not isinstance(reflection, str):
            raise ValueError("Situation and reflection must be strings")
        self._refresh()
        entry = {"situation": situation, "reflection": reflection, "timestamp": int(time.time())}
        if trade_context and isinstance(trade_context, dict):
            entry["trade_context"] = trade_context
        previous = self.memories
        self.memories = previous + [entry]
        try:
            self._save()
        except Exception:
            self.memories = previous
            raise
        
        try:
            self._rebuild_index()
        except Exception as exc:
            self._index = None
            print(f"[Memory] Reflection saved, but semantic indexing failed: {exc}")

    def get_memories(self, query: str, n_matches: int = 2) -> list:
        """Return reflections above the minimum cosine similarity, nearest first."""
        self._refresh()
        if not isinstance(query, str) or not query.strip() or not self.memories or n_matches <= 0:
            return []
        if self._index is None:
            self._rebuild_index()
        vector = self._vectors([query])
        scores, indices = self._index.search(vector, min(int(n_matches), len(self.memories)))
        return [self.memories[int(i)]["reflection"] for score, i in zip(scores[0], indices[0])
                if i >= 0 and float(score) >= self.min_similarity]


### 3b. Trade Reflection Function

In [20]:
# =============================================================================
# Trade Outcome Calculation & Trade-Level Reflection
# =============================================================================

def _safe_float(val):
    if val is None:
        return None
    try:
        f = float(val)
        return f if f == f and f not in (float("inf"), float("-inf")) else None
    except (ValueError, TypeError):
        return None


def calculate_trade_outcome(
    side: str,
    filled_qty: float,
    filled_avg_price: float,
    reflection_price_or_cost_basis: float,
) -> dict:
    """
    Calculate trade P&L and percentage return based on specific trade execution.

    Args:
        side: "buy" or "sell"
        filled_qty: Number of shares actually executed (must be > 0)
        filled_avg_price: Average execution price per share (must be > 0)
        reflection_price_or_cost_basis:
            For buy: later market quote at reflection time
            For sell: pre-trade position cost basis (avg_entry_price)

    Returns:
        dict with:
            outcome_type: "unrealized" (for buy) or "realized" (for sell)
            pnl: float dollar gain/loss
            pnl_pct: float percentage return
            filled_qty: float
            entry_or_cost_basis: float
            outcome_price: float
        or None if inputs are invalid/missing/zero.
    """
    qty = _safe_float(filled_qty)
    fill_price = _safe_float(filled_avg_price)
    ref_or_basis = _safe_float(reflection_price_or_cost_basis)

    if qty is None or qty <= 0:
        return None
    if fill_price is None or fill_price <= 0:
        return None
    if ref_or_basis is None or ref_or_basis <= 0:
        return None

    side_clean = str(side).strip().lower()
    if side_clean == "buy":
        # Mark-to-market unrealized outcome at reflection time
        pnl = (ref_or_basis - fill_price) * qty
        pnl_pct = ((ref_or_basis / fill_price) - 1.0) * 100.0
        return {
            "outcome_type": "unrealized",
            "pnl": round(pnl, 2),
            "pnl_pct": round(pnl_pct, 2),
            "filled_qty": qty,
            "entry_price": fill_price,
            "reflection_price": ref_or_basis,
        }
    elif side_clean == "sell":
        # Realized trade outcome:
        pnl = (fill_price - ref_or_basis) * qty
        pnl_pct = ((fill_price / ref_or_basis) - 1.0) * 100.0
        return {
            "outcome_type": "realized",
            "pnl": round(pnl, 2),
            "pnl_pct": round(pnl_pct, 2),
            "filled_qty": qty,
            "cost_basis": ref_or_basis,
            "exit_price": fill_price,
        }
    else:
        return None


def reflect_on_trade(
    tok,
    model,
    situation: str,
    decision: dict,
    side: str = None,
    filled_qty: float = None,
    entry_or_cost_basis: float = None,
    fill_avg_price: float = None,
    reflection_price: float = None,
    pnl: float = None,
    pnl_pct: float = None,
    outcome_type: str = None,
    quote_source: str = None,
    quote_timestamp: str = None,
    prev_equity: float = None,
    curr_equity: float = None,
) -> str:
    """
    Use LLM to generate a short reflection on a specific trade outcome.

    Supports trade-level outcomes (realized sell P&L or unrealized buy mark-to-market snapshot)
    with backward-compatible support for legacy portfolio equity arguments.

    Returns:
        A short reflection string (max 300 chars).
        Return "No reflection generated." if generation fails.
    """
    try:
        sym = decision.get("symbol", "the stock") if isinstance(decision, dict) else "the stock"
        trade_side = (side or (decision.get("action") if isinstance(decision, dict) else "trade")).lower()

        # trade-level outcome with portfolio equity fallback
        if pnl is not None and pnl_pct is not None and filled_qty is not None:
            is_unrealized = (outcome_type == "unrealized") or (trade_side == "buy")
            outcome_desc = (
                f"Trade Outcome ({'unrealized mark-to-market snapshot' if is_unrealized else 'realized P&L'}): "
                f"{trade_side.upper()} {filled_qty} shares of {sym} | "
                f"P&L: ${pnl:+.2f} ({pnl_pct:+.2f}%). "
            )
            if trade_side == "buy":
                fill_p = fill_avg_price or entry_or_cost_basis or 0.0
                outcome_desc += f"Fill price: ${fill_p:.2f} -> Current mark: ${reflection_price or 0.0:.2f}"
                if quote_source:
                    outcome_desc += f" | Quote source: {quote_source}"
                if quote_timestamp:
                    outcome_desc += f" at {quote_timestamp}"
            else:
                basis_p = entry_or_cost_basis or 0.0
                exit_p = fill_avg_price or reflection_price or 0.0
                outcome_desc += f"Cost basis: ${basis_p:.2f} -> Exit price: ${exit_p:.2f}"
        elif prev_equity is not None and curr_equity is not None:
            pnl = curr_equity - prev_equity
            pnl_pct = (pnl / prev_equity) * 100.0 if prev_equity else 0.0
            outcome_desc = f"Portfolio Outcome: P&L ${pnl:+.2f} ({pnl_pct:+.2f}%), Equity: ${prev_equity:,.2f} -> ${curr_equity:,.2f}"
        else:
            return "No reflection generated."

        gain_loss_word = "gained" if pnl > 0 else ("lost" if pnl < 0 else "broke even")

        system_msg = (
            "You are a trading reflection assistant. Analyze the specific trade outcome "
            "and write a concise 1-2 sentence lesson learned for future reference. "
            "Explicitly mention the stock symbol, the action (buy/sell), and whether the trade gained or lost money. "
            "Note that a short-term mark-to-market movement on a buy is an unrealized snapshot, not finalized profit."
        )

        dec_str = json.dumps(decision) if isinstance(decision, dict) else str(decision)
        user_msg = (
            f"Trade Decision: {dec_str}\n"
            f"{outcome_desc}\n"
            f"Result: Trade {gain_loss_word} ${abs(pnl):.2f}\n"
            f"Situation: {situation}\n\n"
            "Write a concise 1-2 sentence reflection on this trade:"
        )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                eos_token_id=tok.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        reflection = tok.decode(out[0][input_len:], skip_special_tokens=True).strip()

        return reflection[:300] if reflection else "No reflection generated."

    except Exception:
        return "No reflection generated."

### Test Task 3

In [21]:
# Test TradingMemory
try:
    test_mem = TradingMemory("test_memory.json")

    test_mem.add_memory(
        situation="AAPL bullish news, strong earnings, bought 10 shares",
        reflection="Buying on strong earnings news was profitable."
    )
    test_mem.add_memory(
        situation="TSLA bearish sentiment, missed earnings, sold 5 shares",
        reflection="Selling before bearish news helps protect capital."
    )
    test_mem.add_memory(
        situation="MSFT neutral news, market sideways, held position",
        reflection="Holding during uncertain times avoided transaction costs."
    )

    results = test_mem.get_memories("Apple shares climbed after strong quarterly results", n_matches=2)
    print("Semantic memory retrieval test:")
    print(f"  Query: 'Apple shares climbed after strong quarterly results'")
    print(f"  Retrieved {len(results)} memories:")
    for i, r in enumerate(results, 1):
        print(f"    {i}. {r}")

    if "Buying on strong earnings news was profitable." not in results:
        raise AssertionError("Expected the earnings-related semantic memory to be retrieved")

    if os.path.exists("test_memory.json"):
        os.remove("test_memory.json")
    print("\nTradingMemory semantic search works!")
except NotImplementedError:
    print("Task 3a not yet implemented")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

# Test reflect_on_trade
try:
    reflection = reflect_on_trade(
        tok, model,
        situation="AAPL bullish news, bought 5 shares at $280",
        decision={"action": "buy", "symbol": "AAPL", "qty": 5},
        prev_equity=100000.0, curr_equity=100150.0
    )
    print(f"\nReflection test: {reflection}")
    print("reflect_on_trade works!")
except NotImplementedError:
    print("\nTask 3b (reflect_on_trade) not yet implemented")
except Exception as e:
    print(f"\nError: {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17085.08it/s]


Semantic memory retrieval test:
  Query: 'Apple shares climbed after strong quarterly results'
  Retrieved 1 memories:
    1. Buying on strong earnings news was profitable.

TradingMemory semantic search works!

Reflection test: The trade of buying 5 shares in Apple Inc. (AAPL) resulted in a gain of $150.00, reflecting the positive sentiment towards the company's prospects due to bullish news. This experience highlights the potential for profitable trades when timing aligns with market trends.
reflect_on_trade works!


---
## Task 4: [TODO] SFT Fine-tuning with LoRA

### Objective
Fine-tune the Qwen 1.5B model on **backtested trading data** using LoRA (Low-Rank Adaptation), then compare performance.

### Flow

```
Step 1: Backtest with BASE model
         Simulate ~60 trading days on historical prices
         LLM makes decisions each day (no real money)
         All (prompt, decisions, equity_change) logged automatically
                         │
Step 2: Auto-label data
         equity went UP   → label "good"
         equity went DOWN → label "bad"
                         │
Step 3: SFT fine-tune (YOUR IMPLEMENTATION)
         Filter "good" examples
         Tokenize as (prompt, completion) pairs
         LoRA: only train <0.1% of parameters
         Save adapter to Drive
                         │
Step 4: Backtest with FINE-TUNED model
         Same historical data, same starting cash
         Compare returns: base vs fine-tuned
```

### What you need to do

1. **`prepare_sft_dataset()`** - Filter labeled data, tokenize into training format
2. **`get_lora_model()`** - Configure LoRA adapters on the base model
3. **`train_sft()`** - Run the training loop
4. Compare backtest results

### Key Concepts

**LoRA** adds small trainable matrices to attention layers while freezing the base model:
- `r=8`: rank of the low-rank matrices (higher = more capacity but slower)
- `lora_alpha=16`: scaling factor (typically 2x r)
- `target_modules=["q_proj", "v_proj"]`: which layers to adapt
- Only ~0.07% of parameters are trainable → fast training on Colab

**SFT (Supervised Fine-Tuning)**: Train the model to reproduce "good" trading decisions.
The loss is only computed on the **completion** (decisions), not the prompt.

### Grading Criteria
- [ ] `prepare_sft_dataset` correctly filters and tokenizes data; LoRA config is reasonable (r, alpha, target modules); the training loop runs without errors and every epoch's loss is a normal number. *Evidence: 4c output: the dataset size matches the GOOD count (explain any difference in your report), the trainable-parameter line, the loss per epoch.*
- [ ] 4a and 4c are provided cells: run them, then write your observations and explanations in the training-strategy part of your report. The note at the top of 4a and of 4c says what to cover.

### 4a. Run Backtest & Collect Training Data

Run a backtest with the **base model** on historical data. This collects trading decisions and outcomes
without spending any paper money. Data is auto-labeled based on P&L.

> **Provided cell: run it, then explain what you see in your report.** It replays your agent with the base model over the last ~60 trading days (`run_backtest`, 1.14); the collector (1.13) logs the valid days, and `auto_label` marks each one GOOD or BAD from the next day's profit or loss. In your report: how many days were logged, the GOOD/BAD split, whether 1–2 sample labels make sense (the inspect cell below helps), and why you changed the labeling rule if you did.

In [93]:
# =============================================================================
# Run backtest with BASE model to collect training data
# =============================================================================

# Initialize a fresh collector (clears previous data)
import os as _os
_sft_path = _os.path.join(CACHE_DIR, "sft_data.jsonl")
if _os.path.exists(_sft_path):
    _os.remove(_sft_path)
    print("Cleared previous training data")

collector = TradingDataCollector()

# Run backtest - runtime depends on model speed and research turns
print("Running backtest with BASE model...")
print("(Each session uses up to 6 LLM generations; timing and token counts appear below.)\n")
base_results = run_backtest(tok, model, collector=collector, n_days=252)  # 3 months of prices ≈ 63 trading days

# Auto-label: good if equity increased, bad if decreased
collector.auto_label(threshold=0.0)

print("\n--- Training Data Summary ---")
collector.summary()

# Review labeled entries
print("\nRecent entries:")
collector.review(n=10)

# Manual override example (uncomment to use):
# collector.set_label(0, "good")   # override entry 0
# collector.set_label(3, "bad")    # override entry 3

print(f"\nGood examples for training: {len(collector.get_labeled('good'))}")
print(f"Bad examples (not used in SFT): {len(collector.get_labeled('bad'))}")

Cleared previous training data
Running backtest with BASE model...
(Each session uses up to 6 LLM generations; timing and token counts appear below.)

[Backtest setup] Preparing market sessions...
[Backtest setup] Loading historical bars...
[Backtest setup] Ready in 8.7s
BACKTEST 2025-11-06 → 2026-11-06 (252 sessions); decisions after close, fills next open
[Session 1/252] 2025-11-06: starting fills, valuation and research...
  [Generation 1/6] Starting research, attempt 1...
  [Generation] research: 2.3s | input=943 | output=15/180 | cap_hit=False | components={'instructions': 420, 'account_state': 469, 'tool_results': 0, 'assistant_calls': 0, 'phase': 30, 'retry': 0, 'extra_context': 0}
  [Tool] get_rsi {'symbol': 'MSFT'}
  [Tool context] raw=30 tokens; sent=30 tokens; usable=['MSFT']
  [Generation 2/6] Starting decision, attempt 1...
  [Generation] decision: 0.3s | input=1084 | output=2/320 | cap_hit=False | components={'instructions': 420, 'account_state': 469, 'tool_results': 36, 

### Optional: Inspect & Edit SFT Training Data

After the backtest, `sft_data.jsonl` contains all logged (prompt, completion, label) pairs.
Use the cells below to review entries, remove low-quality ones, or upload a hand-edited version
before running `train_sft()`.


In [24]:
# =============================================================================
# OPTIONAL: Inspect & Edit sft_data.jsonl
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

# -- 1. View all SFT entries -----------------------------------------------
if os.path.exists(sft_path):
    with open(sft_path) as f:
        entries = [json.loads(l) for l in f if l.strip()]
    print(f"Found {len(entries)} SFT entries:")
    good = [e for e in entries if e.get('label') == 'good']
    bad  = [e for e in entries if e.get('label') == 'bad']
    print(f"  good: {len(good)}  |  bad: {len(bad)}\n")
    for i, e in enumerate(entries):
        label  = e.get('label', '?')
        pnl    = e.get('pnl', '?')
        compl  = e.get('completion', '')[:100].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      completion: {compl}")
        print()
else:
    print("No sft_data.jsonl found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(sft_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment after editing locally:
# uploaded = files.upload()   # select your edited sft_data.jsonl
# for fname, content in uploaded.items():
#     with open(sft_path, 'wb') as f:
#         f.write(content)
# with open(sft_path) as f:
#     entries = [json.loads(l) for l in f if l.strip()]
# print(f"Reloaded {len(entries)} SFT entries from uploaded file.")

# -- 4. Remove bad entries (keep only good for training) ------------------
# Uncomment to filter:
# if os.path.exists(sft_path):
#     with open(sft_path) as f:
#         entries = [json.loads(l) for l in f if l.strip()]
#     good_only = [e for e in entries if e.get('label') == 'good']
#     with open(sft_path, 'w') as f:
#         for e in good_only:
#             f.write(json.dumps(e) + '\n')
#     print(f"Kept {len(good_only)} good entries; removed {len(entries)-len(good_only)} bad.")


ModuleNotFoundError: No module named 'google.colab'

### 4b. Implement SFT Fine-tuning

In [27]:
# =============================================================================
# TODO: [IMPLEMENTATION REQUIRED] - SFT Fine-tuning with LoRA
# =============================================================================


def prepare_sft_dataset(collector, tok, max_length=2048):
    """Train only clean v2 trades; preserve the exact inference prompt and completion."""
    good_entries = collector.get_labeled("good")
    if not good_entries:
        raise ValueError("No 'good' labeled data found")
    rows = {"input_ids": [], "attention_mask": [], "labels": []}
    dropped = {}
    for entry in good_entries:
        try:
            prompt = validate_sft_example(tok, entry)
            prompt_ids = tok.encode(prompt, add_special_tokens=False)
            full_ids = tok.encode(prompt + entry["completion"] + tok.eos_token, add_special_tokens=False)
            if full_ids[:len(prompt_ids)] != prompt_ids:
                raise ValueError("Tokenizer boundary does not preserve inference prompt")
            if len(full_ids) > max_length:
                raise ValueError("Example exceeds max_length; prompt/completion not truncated")
            labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
            padding = max_length - len(full_ids)
            pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
            rows["input_ids"].append(full_ids + [pad_id] * padding)
            rows["attention_mask"].append([1] * len(full_ids) + [0] * padding)
            rows["labels"].append(labels + [-100] * padding)
        except (ValueError, KeyError, TypeError) as exc:
            key = str(exc)
            dropped[key] = dropped.get(key, 0) + 1
    total_dropped = sum(dropped.values())
    print(f"SFT preparation: kept {len(rows['input_ids'])}/{len(good_entries)}; "
          f"dropped {total_dropped} ({100 * total_dropped / len(good_entries):.1f}%): {dropped}")
    if not rows["input_ids"]:
        raise ValueError("No compatible complete SFT examples; recollect clean v2 episodes or increase max_length")
    return Dataset.from_dict(rows)


def get_lora_model(base_model):
    """
    Wrap the base model with LoRA adapters for fine-tuning.

    Returns:
        PEFT model with LoRA adapters attached.

    TODO:
    1. Create LoraConfig:
       - r=8
       - lora_alpha=16
       - target_modules=["q_proj", "v_proj"]
       - lora_dropout=0.05
       - bias="none"
       - task_type=TaskType.CAUSAL_LM
    2. Apply: peft_model = get_peft_model(base_model, config)
    3. Print trainable params: peft_model.print_trainable_parameters()
    4. Return peft_model
    """
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    peft_model = get_peft_model(base_model, config)
    peft_model.print_trainable_parameters()
    return peft_model


def train_sft(peft_model, dataset, tok, epochs=3, lr=2e-4, batch_size=1):
    """
    Train the LoRA model on the SFT dataset.

    Args:
        peft_model: Model with LoRA adapters
        dataset: Dataset from prepare_sft_dataset()
        tok: Tokenizer
        epochs: Number of training epochs
        lr: Learning rate
        batch_size: Batch size (1 recommended for Colab)

    TODO:
    1. Create optimizer: torch.optim.AdamW(peft_model.parameters(), lr=lr)
    2. Set peft_model.train()
    3. Training loop:
       for epoch in range(epochs):
           total_loss = 0
           for i in range(0, len(dataset), batch_size):
               batch = dataset[i:i+batch_size]
               input_ids = torch.tensor(batch["input_ids"]).to(peft_model.device)
               attention_mask = torch.tensor(batch["attention_mask"]).to(peft_model.device)
               labels = torch.tensor(batch["labels"]).to(peft_model.device)

               outputs = peft_model(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    labels=labels)
               loss = outputs.loss
               loss.backward()
               optimizer.step()
               optimizer.zero_grad()
               total_loss += loss.item()

           avg_loss = total_loss / max(1, len(dataset) // batch_size)
           print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    4. Save adapter:
       save_dir = os.path.join(CACHE_DIR, "lora_adapter")
       peft_model.save_pretrained(save_dir)
       print(f"LoRA adapter saved to {save_dir}")

    5. Set peft_model.eval()
    """
    optimizer = torch.optim.AdamW(peft_model.parameters(), lr=lr)
    peft_model.train()

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        for i in range(0, len(dataset), batch_size):
            batch = dataset[i : i + batch_size]
            input_ids = torch.tensor(batch["input_ids"]).to(peft_model.device)
            attention_mask = torch.tensor(batch["attention_mask"]).to(peft_model.device)
            labels = torch.tensor(batch["labels"]).to(peft_model.device)

            outputs = peft_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(1, n_batches)
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    save_dir = os.path.join(CACHE_DIR, "lora_adapter")
    peft_model.save_pretrained(save_dir)
    print(f"LoRA adapter saved to {save_dir}")

    peft_model.eval()

### 4c. Re-run Backtest & Compare

Run the same backtest with the **fine-tuned model** and compare returns against the base model.

> **Provided cell: run it, then explain what you see in your report.** It trains LoRA with your 4b functions, then backtests the base model and the fine-tuned model on the last 20 trading days and prints the return and number of trades for each. In your report: how the two models differ (return, trades, whether they make real decisions each day), what you think caused it, and which conclusions you would not trust, and why.

In [29]:
# =============================================================================
# Compare base vs fine-tuned model via backtest
# =============================================================================
print("=" * 60)
print("SFT Fine-tuning Pipeline")
print("=" * 60)

collector = TradingDataCollector()
good_count = len(collector.get_labeled("good"))

if good_count == 0:
    print("\nNo 'good' labeled data found!")
    print("Run cell 4a first to backtest and collect training data.")
else:
    print(f"\nTraining on {good_count} 'good' examples...")

    try:
        # Step 1: Prepare dataset
        dataset = prepare_sft_dataset(collector, tok)
        print(f"Dataset ready: {len(dataset)} examples")

        # Step 2: Create LoRA model
        lora_model = get_lora_model(model)

        # Step 3: Train
        train_sft(lora_model, dataset, tok)
        print("\nTraining complete!")

        # Step 4: Backtest base model (comparison baseline)
        # Comment this part if you only want to run Fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Base Model (comparison baseline)")
        print("=" * 60)
        # get_peft_model() adds LoRA to `model` in place, so `model` is no longer
        # the base model here. disable_adapter() turns LoRA off for this baseline run.
        with lora_model.disable_adapter():
            base_results = run_backtest(tok, lora_model, collector=None, n_days=60)

        # Step 5: Backtest with fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Fine-tuned Model")
        print("=" * 60)
        ft_results = run_backtest(tok, lora_model, collector=None, n_days=60)

        # Step 6: Compare
        print("\n" + "=" * 60)
        print("COMPARISON: Base Model vs Fine-tuned Model")
        print("=" * 60)
        print(f"  Base model return:       {base_results['total_return']:>+8.2f}%")
        print(f"  Fine-tuned model return: {ft_results['total_return']:>+8.2f}%")
        diff = ft_results['total_return'] - base_results['total_return']
        print(f"  Improvement:             {diff:>+8.2f}%")
        print(f"")
        print(f"  Base trades:             {len(base_results['trades']):>8d}")
        print(f"  Fine-tuned trades:       {len(ft_results['trades']):>8d}")
        print("=" * 60)

        if diff > 0:
            print("\nFine-tuned model outperformed the base model!")
        elif diff < 0:
            print("\nBase model performed better. Consider:")
            print("  - Collecting more training data (increase n_days)")
            print("  - Adjusting labeling threshold")
            print("  - Tuning LoRA hyperparameters (r, alpha, lr)")
        else:
            print("\nBoth models performed equally.")

    except NotImplementedError as e:
        print(f"\nNot yet implemented: {e}")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc()

SFT Fine-tuning Pipeline
Loaded 92 existing training entries

Training on 46 'good' examples...
SFT preparation: kept 46/46; dropped 0 (0.0%): {}
Dataset ready: 46 examples


c:\Users\albug\anaconda3\envs\nlp\Lib\site-packages\peft\mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\albug\anaconda3\envs\nlp\Lib\site-packages\peft\tuners\tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705
Epoch 1/3 | Avg Loss: 0.3079
Epoch 2/3 | Avg Loss: 0.0827
Epoch 3/3 | Avg Loss: 0.0650
LoRA adapter saved to c:\Users\albug\VS-Projects\LLM-trading-agent\.cache\stock_agent\lora_adapter

Training complete!
BACKTEST: Base Model (comparison baseline)
[Backtest setup] Preparing market sessions...
[Backtest setup] Loading historical bars...
[Backtest setup] Ready in 5.0s
BACKTEST 2026-08-14 → 2026-11-06 (60 sessions); decisions after close, fills next open
[Session 1/60] 2026-08-14: starting fills, valuation and research...
  [Generation 1/6] Starting research, attempt 1...
  [Generation] research: 2.4s | input=943 | output=22/180 | cap_hit=False | components={'instructions': 420, 'account_state': 469, 'tool_results': 0, 'assistant_calls': 0, 'phase': 30, 'retry': 0, 'extra_context': 0}
  [Tool] get_price_history {'symbol': 'AAPL', 'period': '5d'}
  [Tool context] raw=288 tokens; sent=332 tokens; usable=['AAPL']

---
## Task 5: [TODO] Full Trading Loop Integration

### Objective
Wire Tasks 2-3 together with the pre-written infrastructure into a complete trading loop, using the **fine-tuned model** from Task 4.

Each iteration:
1. Get current account state
2. Fetch financial news (from Drive cache)
3. Fetch extra context from your MCP tool functions (Task 2)
4. Check pending trades for reflection (after delay)
5. Retrieve similar past experiences from memory
6. Make LLM-powered decisions
7. Execute orders
8. Queue trades for future reflection

### Important Notes
- The loop runs for `n_iterations` (not infinite)
- Use `time.sleep(LOOP_SECONDS)` between iterations
- `pending_reflections` tracks trades awaiting reflection
- A trade is ready for reflection when `loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS`

### Grading Criteria
- [ ] All steps are correctly wired together; error handling prevents one failed step from crashing the loop; `extra_context` from Task 2 is passed to `llm_agent_decide()`; memory from Task 3 is used for reflections and retrieval; orders are only submitted for buy/sell decisions (not hold). *Evidence: at least 10 consecutive iterations, each showing the memory retrieval and the Task 2 data, and at least one order with its status.*
- [ ] Live week (11/9–11/13): the output of every trading day, plus the Alpaca order history and final balance. *Evidence: the daily logs in `trading_logs/` on Drive; orders and the final balance are checked in your Alpaca account.*

In [30]:
MAX_REFLECTION_RETRIES = 5


class ReflectionPersistenceError(RuntimeError):
    """A pending reflection could not be safely loaded or saved."""


class PendingReflectionStore:
    """Atomically persist pending reflections for one authenticated paper account."""

    def __init__(self, cache_dir, account_id):
        import hashlib
        import math
        if not account_id:
            raise ReflectionPersistenceError("A verified account ID is required")
        account_id = str(account_id)
        self.account_id = account_id
        tag = hashlib.sha256(account_id.encode("utf-8")).hexdigest()[:24]
        self.path = os.path.join(cache_dir, f"pending_reflections_{tag}.json")
        self.items = []
        if os.path.exists(self.path):
            try:
                with open(self.path, "r", encoding="utf-8") as f:
                    payload = json.load(f)
                if payload.get("version") != 1 or payload.get("account_id") != account_id:
                    raise ValueError("Queue version or account ID mismatch")
                items = payload.get("items")
                if not isinstance(items, list) or any(
                    not isinstance(item, dict) or not isinstance(item.get("order_id"), str)
                    or not item["order_id"] or not isinstance(item.get("due_at"), (int, float))
                    or not math.isfinite(item["due_at"]) or not isinstance(item.get("symbol"), str)
                    or not item["symbol"] or item.get("side") not in ("buy", "sell")
                    or not isinstance(item.get("decision"), dict) or not isinstance(item.get("situation"), str)
                    or not isinstance(item.get("retry_count"), int) or item["retry_count"] < 0
                    for item in items
                ):
                    raise ValueError("Invalid pending reflection queue")
                if len({item["order_id"] for item in items}) != len(items):
                    raise ValueError("Duplicate order ID in pending reflection queue")
                self.items = items
            except Exception as exc:
                raise ReflectionPersistenceError(f"Cannot load pending reflections: {exc}") from exc
        self._write(self.items)

    def _write(self, items):
        import tempfile
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=os.path.dirname(self.path),
                                             prefix=".pending_reflections_", suffix=".tmp", delete=False) as f:
                temp_path = f.name
                json.dump({"version": 1, "account_id": self.account_id, "items": items}, f, ensure_ascii=False)
                f.flush()
                os.fsync(f.fileno())
            os.replace(temp_path, self.path)
        except Exception as exc:
            raise ReflectionPersistenceError(f"Cannot save pending reflections: {exc}") from exc
        finally:
            if temp_path and os.path.exists(temp_path):
                os.unlink(temp_path)

    def enqueue(self, item):
        if any(old["order_id"] == item["order_id"] for old in self.items):
            raise ReflectionPersistenceError("Duplicate pending order ID")
        updated = self.items + [item]
        self._write(updated)
        self.items = updated

    def remove(self, order_id):
        updated = [item for item in self.items if item["order_id"] != order_id]
        self._write(updated)
        self.items = updated

    def defer(self, item):
        updated_item = dict(item, retry_count=item.get("retry_count", 0) + 1)
        updated = [updated_item if old["order_id"] == item["order_id"] else old for old in self.items]
        self._write(updated)
        self.items = updated




def _safe_float(val):
    if val is None:
        return None
    try:
        f = float(val)
        return f if f == f else None
    except (ValueError, TypeError):
        return None


def get_current_stock_quote(symbol: str) -> dict:
    """Return a valid price with source and observation-time metadata, or None."""
    if not symbol or not isinstance(symbol, str):
        return None
    sym = symbol.strip().upper()
    retrieved_at = datetime.datetime.now(datetime.timezone.utc).isoformat()
    try:
        url = "https://data.alpaca.markets/v2/stocks/{}/snapshot".format(sym)
        headers = {"APCA-API-KEY-ID": ALPACA_API_KEY, "APCA-API-SECRET-KEY": ALPACA_SECRET_KEY}
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, dict):
                for key, source in (("latestTrade", "alpaca.latest_trade"), ("minuteBar", "alpaca.minute_bar"), ("dailyBar", "alpaca.daily_bar")):
                    item = data.get(key)
                    if isinstance(item, dict):
                        price = _safe_float(item.get("p") if key == "latestTrade" else item.get("c"))
                        if price is not None and price > 0:
                            return {"price": round(price, 2), "source": source, "market_timestamp": item.get("t"), "retrieved_at": retrieved_at}
    except Exception as exc:
        print("  [Price Quote] Alpaca snapshot failed for {}: {}".format(sym, exc))
    try:
        info = getattr(yf.Ticker(sym), "fast_info", None)
        if info and hasattr(info, "last_price"):
            price = _safe_float(info.last_price)
            if price is not None and price > 0:
                return {"price": round(price, 2), "source": "yfinance.fast_info", "market_timestamp": None, "retrieved_at": retrieved_at}
    except Exception as exc:
        print("  [Price Quote] yfinance fallback failed for {}: {}".format(sym, exc))
    return None


def get_current_stock_price(symbol: str) -> float:
    """Backward-compatible price-only wrapper around get_current_stock_quote()."""
    quote = get_current_stock_quote(symbol)
    return quote["price"] if quote else None


def run_trading_loop(tc, tok, model, memory=None, collector=None,
                     n_iterations=5):
    """
    Execute the main trading loop.

    Args:
        tc: Alpaca TradingClient
        tok: Tokenizer
        model: LLM model (use fine-tuned lora_model from Task 4)
        memory: TradingMemory instance (None to disable memory)
        collector: TradingDataCollector instance (None to disable data collection)
        n_iterations: Number of loop iterations to run
    """
    account_id = verify_paper_account(tc)
    reflection_store = PendingReflectionStore(CACHE_DIR, account_id)
    pending_outcomes = []
    print(f"Paper endpoint verified; restored {len(reflection_store.items)} pending reflections.")

    for loop_count in range(1, n_iterations + 1):
        print(f"\n{'='*60}")
        print(f"LOOP #{loop_count} / {n_iterations} | Pending reflections: {len(reflection_store.items)}")
        print('='*60)

        #Get current account state
        try:
            state = get_state(tc)
            print(f"Account State: Equity: ${state['equity']:,.2f} | Cash: ${state['cash']:,.2f} | Buying Power: ${state['buying_power']:,.2f}")
            if state.get("positions"):
                pos_str = ", ".join(f"{p['symbol']} ({p['qty']} sh @ ${p['avg_entry_price'] or 0:.2f})" for p in state["positions"])
                print(f"Current Positions: {pos_str}")
            else:
                print("Current Positions: None")
        except Exception as e:
            print(f"[Safety Halt] Error fetching account state from Alpaca: {e}")
            print("  Skipping loop iteration to prevent executing orders on unverified or fabricated account state.")
            if loop_count < n_iterations:
                print(f"\nSleeping {LOOP_SECONDS}s before next loop...")
                time.sleep(LOOP_SECONDS)
            continue

        held_symbols = [p["symbol"] for p in state.get("positions", [])]
        focus_symbols = list(dict.fromkeys(held_symbols + [s for s in WHITELIST[:5]]))[:5]

        # Fetch financial news
        news_context = ""
        try:
            news_context = get_news_for_symbols(focus_symbols)
            preview = news_context[:250].replace("\n", " ")
            print(f"News fetched for {focus_symbols}: {preview}...")
        except Exception as e:
            print(f"Error fetching news: {e}")

        # Fetch extra context from your MCP tools (Task 2)
        extra_context = ""
        try:
            extra_context = get_extra_context(focus_symbols)
            if extra_context:
                print(f"Extra context (Task 2 tools) collected for {focus_symbols}:")
                for line in extra_context.split("\n")[:8]:
                    print(f"  {line}")
                if len(extra_context.split("\n")) > 8:
                    print("  ...")
            else:
                print("No extra context returned.")
        except Exception as e:
            print(f"Error fetching extra context: {e}")

        # Step 4: Check pending reflections
        ready_reflections = [trade for trade in reflection_store.items
                             if time.time() >= trade["due_at"]]

        quote_fn = globals().get("get_current_stock_quote", get_current_stock_quote)

        for trade in ready_reflections:
            if memory is None:
                continue

            order_id = trade.get("order_id")
            if any(isinstance(m, dict) and isinstance(m.get("trade_context"), dict)
                   and m["trade_context"].get("order_id") == order_id
                   and m["trade_context"].get("account_id") in (None, account_id)
                   for m in memory.memories):
                reflection_store.remove(order_id)
                continue
            symbol = trade.get("symbol")
            side = trade.get("side", "buy").lower()
            submitted_qty = trade.get("submitted_qty", 0)

            order_obj = None
            if order_id and tc is not None:
                try:
                    order_obj = tc.get_order_by_id(order_id)
                except Exception as e:
                    print(f"  [Reflection Warning] Error checking order {order_id}: {e}")

            if order_obj is None:
                if trade.get("retry_count", 0) < MAX_REFLECTION_RETRIES:
                    reflection_store.defer(trade)
                    print(f"  [Reflection Deferred] Retrying order {order_id} ({trade.get('retry_count', 0) + 1}/{MAX_REFLECTION_RETRIES})")
                else:
                    print(f"  [Reflection Discarded] Order {order_id} unreachable after {MAX_REFLECTION_RETRIES} attempts.")
                    reflection_store.remove(order_id)
                continue

            raw_status = getattr(order_obj, "status", "")
            curr_status = str(getattr(raw_status, "value", raw_status)).lower().split(".")[-1]
            filled_qty = _safe_float(getattr(order_obj, "filled_qty", None)) or 0.0
            filled_avg_price = _safe_float(getattr(order_obj, "filled_avg_price", None)) or 0.0

            terminal_statuses = {"filled", "canceled", "expired", "rejected", "replaced"}
            is_terminal = curr_status in terminal_statuses
            if not is_terminal:
                if trade.get("retry_count", 0) < MAX_REFLECTION_RETRIES:
                    reflection_store.defer(trade)
                    print(f"  [Reflection Deferred] Order {order_id} is '{curr_status}' (filled: {filled_qty}/{submitted_qty}); deferring to next loop.")
                else:
                    print(f"  [Reflection Skipped] Order {order_id} remained nonterminal ('{curr_status}') after {MAX_REFLECTION_RETRIES} retries; no reflection stored.")
                    reflection_store.remove(order_id)
                continue

            if filled_qty <= 0:
                print(f"  [Reflection Discarded] Order {order_id} ended with 0 shares filled (status: {curr_status}).")
                reflection_store.remove(order_id)
                continue

            if side == "buy":
                quote = quote_fn(symbol)
                if isinstance(quote, dict):
                    current_quote = _safe_float(quote.get("price"))
                    quote_source = quote.get("source")
                    quote_timestamp = quote.get("market_timestamp") or quote.get("retrieved_at")
                else:
                    current_quote = _safe_float(quote)
                    quote_source = "legacy_quote_provider"
                    quote_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
                if current_quote is None or current_quote <= 0:
                    if trade.get("retry_count", 0) < MAX_REFLECTION_RETRIES:
                        reflection_store.defer(trade)
                        print(f"  [Reflection Deferred] Could not fetch current quote for {symbol}; deferring reflection.")
                        continue
                    else:
                        print(f"  [Reflection Skipped] Quote unavailable for {symbol} after {MAX_REFLECTION_RETRIES} attempts; skipping.")
                        reflection_store.remove(order_id)
                        continue

                outcome = calculate_trade_outcome(
                    side="buy",
                    filled_qty=filled_qty,
                    filled_avg_price=filled_avg_price,
                    reflection_price_or_cost_basis=current_quote,
                )
            else:  # sell
                cost_basis = trade.get("cost_basis")
                quote_source = "alpaca.order_fill"
                filled_at = getattr(order_obj, "filled_at", None)
                quote_timestamp = str(filled_at) if filled_at is not None else None
                if cost_basis is None or cost_basis <= 0:
                    print(f"  [Reflection Skipped] Missing cost basis for sell of {symbol}; skipping.")
                    reflection_store.remove(order_id)
                    continue

                outcome = calculate_trade_outcome(
                    side="sell",
                    filled_qty=filled_qty,
                    filled_avg_price=filled_avg_price,
                    reflection_price_or_cost_basis=cost_basis,
                )

            if not outcome:
                print(f"  [Reflection Skipped] Invalid outcome calculation for {symbol} (filled_qty={filled_qty}, fill_price={filled_avg_price}).")
                reflection_store.remove(order_id)
                continue

            # Generate reflection
            try:
                reflection = reflect_on_trade(
                    tok, model,
                    situation=trade["situation"],
                    decision=trade["decision"],
                    side=side,
                    filled_qty=outcome["filled_qty"],
                    entry_or_cost_basis=outcome.get("entry_price") or outcome.get("cost_basis"),
                    fill_avg_price=filled_avg_price,
                    reflection_price=outcome.get("reflection_price") or outcome.get("exit_price"),
                    pnl=outcome["pnl"],
                    pnl_pct=outcome["pnl_pct"],
                    outcome_type=outcome["outcome_type"],
                    quote_source=quote_source,
                    quote_timestamp=quote_timestamp,
                )

                outcome.update({"symbol": symbol, "side": side, "account_id": account_id, "order_id": str(order_id), "fill_avg_price": filled_avg_price, "quote_source": quote_source, "quote_timestamp": quote_timestamp})
                outcome_label = str(outcome['outcome_type']).upper()
                reflection_record = (
                    f"[{symbol} {side.upper()} ({outcome_label}): "
                    f"P&L ${outcome['pnl']:+.2f} ({outcome['pnl_pct']:+.2f}%) on {outcome['filled_qty']} sh] "
                    f"{reflection}"
                )
                if reflection == "No reflection generated.":
                    raise RuntimeError("Reflection generation failed")
                memory.add_memory(trade["situation"], reflection_record, trade_context=outcome)
                reflection_store.remove(order_id)
                print(f"[Reflection stored]: {reflection_record}")
            except Exception as e:
                raise RuntimeError(f"Reflection processing failed for {order_id}; queue retained") from e

        # Retrieve relevant past reflections
        past_reflections = []
        pos_summary = ", ".join(f"{p['symbol']} ({p['qty']} sh)" for p in state.get("positions", [])) or "None"
        situation_summary = (
            f"Positions: {pos_summary}. Cash: ${state['cash']:,.2f}, Equity: ${state['equity']:,.2f}. "
            f"Active tickers: {', '.join(focus_symbols)}."
        )
        if news_context:
            situation_summary += f" News: {news_context[:200].replace(chr(10), ' ')}"

        if memory is not None:
            try:
                past_reflections = memory.get_memories(situation_summary, n_matches=MEMORY_TOP_K)
                print(f"Memory Retrieval ({len(past_reflections)} matches):")
                if past_reflections:
                    for i, ref in enumerate(past_reflections, 1):
                        print(f"  {i}. {ref}")
                else:
                    print("  No memories met the semantic similarity threshold.")
            except Exception as e:
                print(f"Error retrieving memory: {e}")
                past_reflections = []
        else:
            print("Memory disabled (memory is None).")

        # Make trading decisions
        decisions = []
        sft_prompt = ""
        sft_completion = ""
        tool_calls_made = 0
        agent_metadata = None
        try:
            agent_result = llm_agent_decide(
                tok, model, state, universe=WHITELIST,
                extra_context=extra_context,
                past_reflections=past_reflections
            )
            decisions, sft_prompt, sft_completion, tool_calls_made = agent_result
            agent_metadata = getattr(agent_result, "diagnostics", None)
            print(f"\nDecisions ({len(decisions)}):")
            for d in decisions:
                action = d.get("action", "hold").upper()
                sym = d.get("symbol", "?")
                qty = d.get("qty", 0)
                reason = d.get("reason", "")
                reason_str = f" | reason: {reason}" if reason else ""
                print(f"  - {sym}: {action} {qty} shares{reason_str}")
        except Exception as e:
            print(f"Error in llm_agent_decide: {e}")
            decisions = [{"action": "hold", "symbol": s, "qty": None, "reason": f"error: {e}"} for s in WHITELIST[:5]]

        # Execute orders
        for d in decisions:
            action = d.get("action", "hold")
            symbol = d.get("symbol")
            qty = d.get("qty")
            if action in ("buy", "sell"):
                try:
                    # Look up pre-trade cost basis if selling
                    cost_basis = None
                    if action == "sell":
                        for p in state.get("positions", []):
                            if p.get("symbol") == symbol:
                                cost_basis = _safe_float(p.get("avg_entry_price"))
                                break
                        if cost_basis is None or cost_basis <= 0:
                            print(f"  [Notice] No valid cost basis found for {symbol} prior to sell; will not be eligible for reflection.")

                    if "order_type" not in d:
                        d["order_type"] = "market"
                    order_result = submit_order(tc, d, expected_account_id=account_id)
                    qty = order_result["qty"]
                    status = str(order_result.get("status", "")).lower()
                    order_id = order_result.get("id")
                    print(f"  [Order Submitted] {action.upper()} {qty} {symbol} -> Status: {status} (ID: {order_id})")

                    if order_id:
                        if action == "sell" and (cost_basis is None or cost_basis <= 0):
                            print(f"  [Reflection Skipped] Cannot queue sell reflection without cost basis for {symbol}")
                        else:
                            submitted_at = time.time()
                            reflection_store.enqueue({
                                "timestamp": submitted_at,
                                "due_at": submitted_at + max(0, REFLECTION_DELAY_LOOPS * LOOP_SECONDS),
                                "situation": situation_summary,
                                "decision": d,
                                "order_id": order_id,
                                "symbol": symbol,
                                "side": action,
                                "submitted_qty": qty,
                                "cost_basis": cost_basis,
                                "retry_count": 0,
                            })
                    else:
                        print(f"  [Order Skipped for Reflection] No order ID for {symbol} (status: {status})")
                except ReflectionPersistenceError:
                    print(f"  [Safety Halt] Could not persist reflection for {symbol}; check broker orders before restarting.")
                    raise
                except Exception as e:
                    print(f"  [Order Error] {action.upper()} {symbol}: {e}")
            else:
                reason = d.get("reason", "hold")
                print(f"  [Hold] {symbol}: {reason}")

        if collector is not None and sft_prompt and sft_completion:
            try:
                idx = collector.log(sft_prompt, _trade_completion(decisions), decisions,
                                    state["equity"], tool_calls_made, metadata=agent_metadata)
                if idx is not None:
                    pending_outcomes.append({"idx": idx, "loop": loop_count})
            except Exception as e:
                print(f"Error logging to collector: {e}")

        # Sleep
        if loop_count < n_iterations:
            print(f"\nSleeping {LOOP_SECONDS}s before next loop...")
            time.sleep(LOOP_SECONDS)

### Run the Trading Loop

Use the **fine-tuned model** (`lora_model`) from Task 4 for live paper trading.

#### Live week (11/9–11/13)

- Each trading day, run the loop during market hours (9:30–16:00 ET), ideally for 2–4 hours (longer if you can; set `LOOP_SECONDS` and `n_iterations` to match).
- **Do not rerun 4a or 4c on live days.** 4a clears your training data and 4c retrains the model, which would change your frozen model. Run Task 1 to Task 3 and the `run_trading_loop` cell, then the cell below: it loads the adapter Task 4 saved on Drive by itself.
- **The daily log is kept for you.** Everything the cell below prints is also saved to `CACHE_DIR/trading_logs/<date>.log` on Drive, so make your loop print each decision and order clearly. You submit these logs on 11/19, and your report reflects on the week.

In [ ]:
# Initialize memory system
try:
    memory = TradingMemory(os.path.join(CACHE_DIR, "trading_memory.json"))  # keep on Drive
    print(f"Memory loaded: {len(memory.memories)} past entries")
except Exception as e:
    memory = None
    print(f"Semantic memory unavailable; retrieval disabled: {e}")

# Initialize data collector
collector = TradingDataCollector()

# Use fine-tuned model from Task 4. On live days (4a and 4c skipped), load the adapter Task 4 saved on Drive.
try:
    trading_model = lora_model
    model_note = "fine-tuned LoRA model from Task 4 (this session)"
    print("Using fine-tuned LoRA model from Task 4")
except NameError:
    try:
        from peft import PeftModel
        trading_model = PeftModel.from_pretrained(model, os.path.join(CACHE_DIR, "lora_adapter"))
        trading_model.eval()
        model_note = "fine-tuned LoRA adapter loaded from Drive"
        print("Loaded the fine-tuned LoRA adapter saved on Drive")
    except Exception as e:
        trading_model = model
        model_note = f"BASE model (no saved LoRA adapter: {type(e).__name__})"
        print(f"WARNING: no saved LoRA adapter ({type(e).__name__}), using the BASE model. Run Task 4 first.")

# Daily trading log: everything printed below also goes to a file on Drive.
import contextlib, sys
log_dir = os.path.join(CACHE_DIR, "trading_logs")
os.makedirs(log_dir, exist_ok=True)
now_utc = datetime.datetime.now(datetime.timezone.utc)
log_path = os.path.join(log_dir, f"{now_utc.date().isoformat()}.log")

class _Tee:
    """Writes to the notebook output and to the log file at the same time."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
            s.flush()
        return len(text)
    def flush(self):
        for s in self.streams:
            s.flush()

with open(log_path, "a", encoding="utf-8") as log_file, contextlib.redirect_stdout(_Tee(sys.stdout, log_file)):
    print(f"\n===== Run started {now_utc.strftime('%Y-%m-%d %H:%M')} UTC =====")
    print(f"  Model: {model_note}")
    print("\nStarting trading loop...")
    print(f"  Symbols: {len(WHITELIST)} stocks")
    print(f"  Iterations: 80")
    print(f"  Interval: {LOOP_SECONDS}s")
    print("-" * 60)

    try:
        run_trading_loop(tc, tok, trading_model, memory=memory,
                         collector=collector, n_iterations=80)
        print("\n\nTrading loop completed successfully!")
        print("\nData collected:")
        collector.summary()
    except NotImplementedError:
        print("Task 5 not yet implemented")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc(file=sys.stdout)

print(f"\nTrading log saved to {log_path}")


### Optional: Inspect & Edit Memory

Use these cells to view, manually edit, download, or upload `trading_memory.json`.
You can add your own reflections or remove bad ones before the trading loop uses them.


In [ ]:
# =============================================================================
# OPTIONAL: Inspect & Edit trading_memory.json
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

memory_path = os.path.join(CACHE_DIR, "trading_memory.json")

# -- 1. View all memory entries --------------------------------------------
if os.path.exists(memory_path):
    with open(memory_path) as f:
        memories = json.load(f)
    print(f"Found {len(memories)} memory entries:\n")
    for i, m in enumerate(memories):
        label = m.get('label', '?')
        pnl   = m.get('pnl', '?')
        refl  = m.get('reflection', '')[:120].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      {refl}")
        print()
else:
    print("No trading_memory.json found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(memory_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment the block below after you have edited the file locally:
# uploaded = files.upload()   # select your edited trading_memory.json
# for fname, content in uploaded.items():
#     with open(memory_path, 'wb') as f:
#         f.write(content)
# with open(memory_path) as f:
#     memories = json.load(f)
# print(f"Reloaded {len(memories)} memories from uploaded file.")

# -- 4. Delete a bad entry by index ---------------------------------------
# Example: remove entry at index 2
# BAD_IDX = 2
# if os.path.exists(memory_path):
#     with open(memory_path) as f:
#         memories = json.load(f)
#     removed = memories.pop(BAD_IDX)
#     with open(memory_path, 'w') as f:
#         json.dump(memories, f, indent=2)
#     print(f"Removed entry {BAD_IDX}: {removed.get('reflection','')[:60]}")
#     print(f"{len(memories)} entries remain.")


---
## Optional Challenges

### 1. DPO (Direct Preference Optimization)
Instead of SFT (training only on "good" examples), use **DPO** to train on preference pairs:

```python
from trl import DPOTrainer, DPOConfig

# Format: (prompt, chosen_response, rejected_response)
# "chosen" = decisions labeled "good"
# "rejected" = decisions labeled "bad"

dpo_dataset = Dataset.from_dict({
    "prompt": [e["prompt"] for e in pairs],
    "chosen": [e["good_completion"] for e in pairs],
    "rejected": [e["bad_completion"] for e in pairs],
})

config = DPOConfig(
    output_dir="dpo_output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
)

trainer = DPOTrainer(
    model=lora_model,
    ref_model=None,        # with a LoRA model, trl uses it with the adapter off as the reference
    train_dataset=dpo_dataset,
    processing_class=tok,
    args=config,
)
trainer.train()
```

**Key difference**: SFT teaches "do this", DPO teaches "prefer this over that" — generally produces better-calibrated models.

### 2. Vector Database RAG
Replace BM25 with semantic search:
- Use sentence embeddings (`sentence-transformers`)
- Store in FAISS vector index
- Retrieve by cosine similarity

### 3. Advanced Trading Strategy
- Position sizing based on confidence scores
- Stop-loss and take-profit logic
- Portfolio performance tracking with matplotlib

### 4. More Data Sources
- SEC filings (EDGAR API)
- Social media sentiment (Reddit, Twitter)
- Options chain data
- Earnings calendar integration